# Model Training

## Purpose

This notebook develops, evaluates, explains, and saves machine learning models for schedule-time flight-delay prediction using the engineered `flights_features` dataset.

The target variable is `ARR_DEL15`, where a value of `1` indicates that a flight arrived at least 15 minutes late and a value of `0` indicates that it did not.

The workflow is designed to preserve temporal ordering, prevent target leakage, support reproducible model comparison, and produce both prediction and explainability artifacts for the operational dashboard.

## Notebook Workflow

1. **Load and validate the engineered feature dataset**
   - Read the `flights_features` Delta table.
   - Validate required columns, target values, and date coverage.
   - Confirm that the dataset is suitable for model development.

2. **Create chronological train, validation, and test splits**
   - Training period: January–August 2025.
   - Validation period: September–October 2025.
   - Holdout test period: November–December 2025.
   - Keep the final test dataset completely isolated until final evaluation.

3. **Engineer leakage-safe historical predictors**
   - Historical airline delay rate.
   - Historical origin-airport delay rate.
   - Historical destination-airport delay rate.
   - Historical route delay rate.
   - Calculate training-period history using prior observations only.
   - Apply training-derived mappings and fallback rates to later datasets.

4. **Create model-ready feature vectors**
   - Separate categorical and numerical predictors.
   - Use Spark's `FeatureHasher` to generate a fixed-size sparse feature vector.
   - Apply the same deterministic transformation to training, validation, and test datasets.
   - Avoid Spark Connect model-cache limitations associated with large fitted encoding pipelines.

5. **Train and evaluate baseline candidate models**
   - Majority Class Baseline.
   - Logistic Regression.
   - Random Forest.
   - Gradient-Boosted Trees.
   - Calculate overall classification metrics.
   - Calculate delayed-flight positive-class metrics.
   - Generate confusion matrices and baseline comparisons.

6. **Perform chronological hyperparameter tuning**
   - Create four expanding-window validation folds.
   - Use a stratified tuning sample to reduce computational cost and improve delayed-flight representation.
   - Tune Logistic Regression, Random Forest, and Gradient-Boosted Trees.
   - Compare candidate configurations using delayed-flight Recall, delayed-flight F1-score, PR AUC, ROC AUC, and training time.

7. **Compare tuned model performance**
   - Select the best configuration from each algorithm.
   - Compare overall performance and delayed-flight detection performance.
   - Assess operational trade-offs between accuracy, Recall, Precision, and ranking metrics.

8. **Select the final predictive model**
   - Retain Logistic Regression as the selected algorithm.
   - Use the tuned hyperparameters identified through chronological validation.
   - Document the model-selection rationale.

9. **Retrain, evaluate, and save the final model**
   - Retrain the selected Logistic Regression model using the combined training and validation periods.
   - Preserve the stratified sampling strategy used during tuning.
   - Evaluate the retrained model on the untouched holdout test dataset.
   - Save the final model in native Spark ML format.
   - Save supporting metadata and evaluation metrics.

10. **Generate SHAP explainability outputs**
    - Create a separate human-readable explainability workflow.
    - Train an interpretable surrogate Logistic Regression model.
    - Compute SHAP values.
    - Generate:
      - Ranked global feature importance.
      - SHAP summary plot.
      - Direction of feature effects.
      - Dependence plots for important variables.
      - Local waterfall explanations for individual flights.
    - Include the required disclaimer that SHAP explains model behavior and does not prove causation.

## Final Outputs

The notebook produces:

- A saved Spark ML Logistic Regression model.
- Final holdout evaluation metrics.
- Model metadata and preprocessing information.
- Tuned model comparison results.
- Global SHAP explainability artifacts.
- Local flight-level SHAP explanations.
- Dashboard-ready prediction and explainability outputs.

## 1. Load and Validate the Feature Dataset

The model-training process begins by loading the managed `flights_features` Delta table produced by the Feature Engineering notebook.

Before splitting or modelling, the dataset is validated to confirm that:

- The required Unity Catalog table exists
- The target variable is available
- The flight date is stored as a valid date
- All required schedule-time predictors are present
- The dataset contains records suitable for chronological splitting

In [0]:
from __future__ import annotations

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T


FEATURE_TABLE = "workspace.default.flights_features"
TARGET_COLUMN = "ARR_DEL15"
DATE_COLUMN = "FL_DATE"


def require_table(table_name: str) -> None:
    """Raise an error when a required Unity Catalog table is unavailable."""
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Required table '{table_name}' was not found. "
            "Run the feature-engineering notebook before continuing."
        )


require_table(FEATURE_TABLE)

df_features: DataFrame = spark.table(FEATURE_TABLE)

required_columns = {
    "QUARTER",
    "MONTH",
    "DAY_OF_WEEK",
    "FL_DATE",
    "OP_UNIQUE_CARRIER",
    "ORIGIN",
    "DEST",
    "DISTANCE",
    "CRS_DEP_TIME",
    "CRS_ARR_TIME",
    "CRS_ELAPSED_TIME",
    "DEP_HOUR",
    "DEP_MINUTE",
    "IS_WEEKEND",
    "SEASON",
    "TIME_OF_DAY",
    "FLIGHT_DISTANCE_CATEGORY",
    "ARR_DEL15",
}

missing_columns = sorted(required_columns - set(df_features.columns))

if missing_columns:
    raise ValueError(
        "Model-training validation failed. "
        f"Missing required columns: {missing_columns}"
    )

feature_row_count = df_features.count()
feature_column_count = len(df_features.columns)

date_type = df_features.schema[DATE_COLUMN].dataType

if not isinstance(date_type, T.DateType):
    raise TypeError(
        f"{DATE_COLUMN} must be a Spark date column, "
        f"but found {date_type.simpleString()}."
    )

print("Feature dataset loaded and validated successfully.")
print(f"Source table: {FEATURE_TABLE}")
print(f"Total records: {feature_row_count:,}")
print(f"Total columns: {feature_column_count}")
print(f"Prediction target: {TARGET_COLUMN}")
print(f"Date column type: {date_type.simpleString()}")

### 1.1 Date Range and Target Distribution

Before defining the chronological training, validation, and test periods, the feature dataset is examined to confirm its available date range and the distribution of the binary target variable.

This review supports two important modelling decisions:

- Selecting non-overlapping chronological split periods
- Assessing whether the delayed and on-time classes are imbalanced

No records are modified during this analysis.

In [0]:
dataset_profile = (
    df_features
    .select(
        F.min("FL_DATE").alias("MIN_FL_DATE"),
        F.max("FL_DATE").alias("MAX_FL_DATE"),
        F.count("*").alias("TOTAL_RECORDS"),
        F.sum(
            F.when(F.col("ARR_DEL15") == 0, 1).otherwise(0)
        ).alias("ON_TIME_RECORDS"),
        F.sum(
            F.when(F.col("ARR_DEL15") == 1, 1).otherwise(0)
        ).alias("DELAYED_RECORDS"),
    )
    .withColumn(
        "ON_TIME_PERCENTAGE",
        F.round(
            F.col("ON_TIME_RECORDS") / F.col("TOTAL_RECORDS") * 100,
            4,
        ),
    )
    .withColumn(
        "DELAYED_PERCENTAGE",
        F.round(
            F.col("DELAYED_RECORDS") / F.col("TOTAL_RECORDS") * 100,
            4,
        ),
    )
)

display(dataset_profile)

## 2. Create the Chronological Train, Validation, and Test Split

The dataset is divided chronologically rather than randomly because the model is intended to predict future flight-delay risk from historical observations.

The split periods are defined as follows:

- **Training period:** January 1, 2025 to August 31, 2025
- **Validation period:** September 1, 2025 to October 31, 2025
- **Test period:** November 1, 2025 to December 31, 2025

This design ensures that later flight outcomes are not used to train models evaluated on earlier periods. The validation dataset will support model and hyperparameter selection, while the test dataset will remain untouched until final evaluation.

In [0]:
TRAIN_END_DATE = "2025-08-31"
VALIDATION_START_DATE = "2025-09-01"
VALIDATION_END_DATE = "2025-10-31"
TEST_START_DATE = "2025-11-01"

df_train = df_features.filter(
    F.col("FL_DATE") <= F.to_date(F.lit(TRAIN_END_DATE))
)

df_validation = df_features.filter(
    (F.col("FL_DATE") >= F.to_date(F.lit(VALIDATION_START_DATE)))
    & (F.col("FL_DATE") <= F.to_date(F.lit(VALIDATION_END_DATE)))
)

df_test = df_features.filter(
    F.col("FL_DATE") >= F.to_date(F.lit(TEST_START_DATE))
)

split_summary = (
    df_train.select(
        F.lit("TRAIN").alias("DATASET"),
        F.min("FL_DATE").alias("MIN_DATE"),
        F.max("FL_DATE").alias("MAX_DATE"),
        F.count("*").alias("TOTAL_RECORDS"),
        F.avg(F.col("ARR_DEL15").cast("double")).alias("DELAY_RATE"),
    )
    .unionByName(
        df_validation.select(
            F.lit("VALIDATION").alias("DATASET"),
            F.min("FL_DATE").alias("MIN_DATE"),
            F.max("FL_DATE").alias("MAX_DATE"),
            F.count("*").alias("TOTAL_RECORDS"),
            F.avg(F.col("ARR_DEL15").cast("double")).alias("DELAY_RATE"),
        )
    )
    .unionByName(
        df_test.select(
            F.lit("TEST").alias("DATASET"),
            F.min("FL_DATE").alias("MIN_DATE"),
            F.max("FL_DATE").alias("MAX_DATE"),
            F.count("*").alias("TOTAL_RECORDS"),
            F.avg(F.col("ARR_DEL15").cast("double")).alias("DELAY_RATE"),
        )
    )
    .withColumn(
        "DELAY_PERCENTAGE",
        F.round(F.col("DELAY_RATE") * 100, 4),
    )
    .drop("DELAY_RATE")
)

display(split_summary)

### 2.1 Split Validation Summary

The chronological split produced three non-overlapping datasets whose combined record count matches the complete feature dataset.

The target distribution varies across the periods:

- The training period has a delay rate of approximately 22.91%.
- The validation period has a lower delay rate of approximately 18.55%.
- The test period has a higher delay rate of approximately 23.71%.

This variation reflects temporal changes in airline operations and confirms the importance of evaluating the model on future periods rather than using a random split.

## 3. Engineer Leakage-Safe Historical Features

Historical performance features summarize prior delay behaviour for airlines, airports, and routes.

To prevent target leakage:

- Historical features for training records use only flights from earlier dates.
- The current record and later training outcomes are excluded.
- Validation and test mappings will be calculated from the training period only.
- A global training delay rate will be used when insufficient historical observations are available.

The first feature created is `AIRLINE_HIST_DELAY_RATE`, representing an airline's smoothed arrival-delay rate before the current flight date.

In [0]:
from pyspark.sql.window import Window


# Daily airline-level delay statistics within the training period
airline_daily_stats = (
    df_train
    .groupBy(
        "OP_UNIQUE_CARRIER",
        "FL_DATE",
    )
    .agg(
        F.count("*").alias("DAILY_FLIGHT_COUNT"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias("DAILY_DELAY_COUNT"),
    )
)

# Use only dates before the current flight date
airline_history_window = (
    Window
    .partitionBy("OP_UNIQUE_CARRIER")
    .orderBy(F.col("FL_DATE").cast("timestamp").cast("long"))
    .rowsBetween(Window.unboundedPreceding, -1)
)

airline_daily_history = (
    airline_daily_stats
    .withColumn(
        "AIRLINE_PRIOR_FLIGHTS",
        F.sum("DAILY_FLIGHT_COUNT").over(airline_history_window),
    )
    .withColumn(
        "AIRLINE_PRIOR_DELAYS",
        F.sum("DAILY_DELAY_COUNT").over(airline_history_window),
    )
)

display(
    airline_daily_history
    .select(
        "OP_UNIQUE_CARRIER",
        "FL_DATE",
        "DAILY_FLIGHT_COUNT",
        "DAILY_DELAY_COUNT",
        "AIRLINE_PRIOR_FLIGHTS",
        "AIRLINE_PRIOR_DELAYS",
    )
    .orderBy(
        "OP_UNIQUE_CARRIER",
        "FL_DATE",
    )
    .limit(30)
)

### 3.1 Historical Airline Delay Rate

The `AIRLINE_HIST_DELAY_RATE` feature represents an airline's arrival-delay rate using only flights from earlier training dates.

A smoothed estimate is used to prevent unstable rates when an airline has limited prior observations. The overall training delay rate serves as the prior and as the fallback value for the first available date, when no earlier airline history exists.

The current date's outcomes are excluded from the calculation.

In [0]:
# Overall delay rate from the training period.
# This is used as the smoothing prior and first-date fallback.
global_training_delay_rate = (
    df_train
    .select(F.avg(F.col("ARR_DEL15").cast("double")).alias("GLOBAL_DELAY_RATE"))
    .first()["GLOBAL_DELAY_RATE"]
)

# Controls how strongly low-volume airline histories are pulled
# toward the global training delay rate.
SMOOTHING_STRENGTH = 100.0

airline_history_features = (
    airline_daily_history
    .withColumn(
        "AIRLINE_HIST_DELAY_RATE",
        (
            F.coalesce(
                F.col("AIRLINE_PRIOR_DELAYS").cast("double"),
                F.lit(0.0),
            )
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.coalesce(
                F.col("AIRLINE_PRIOR_FLIGHTS").cast("double"),
                F.lit(0.0),
            )
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "OP_UNIQUE_CARRIER",
        "FL_DATE",
        "AIRLINE_PRIOR_FLIGHTS",
        "AIRLINE_PRIOR_DELAYS",
        "AIRLINE_HIST_DELAY_RATE",
    )
)

# Join the leakage-safe airline history to every training flight.
df_train_hist = (
    df_train
    .join(
        airline_history_features,
        on=["OP_UNIQUE_CARRIER", "FL_DATE"],
        how="left",
    )
    .withColumn(
        "AIRLINE_HIST_DELAY_RATE",
        F.coalesce(
            F.col("AIRLINE_HIST_DELAY_RATE"),
            F.lit(global_training_delay_rate),
        ),
    )
)

print(f"Global training delay rate: {global_training_delay_rate:.6f}")
print(f"Smoothing strength: {SMOOTHING_STRENGTH:.0f}")
print(f"Training rows after join: {df_train_hist.count():,}")

display(
    df_train_hist
    .select(
        "OP_UNIQUE_CARRIER",
        "FL_DATE",
        "AIRLINE_PRIOR_FLIGHTS",
        "AIRLINE_PRIOR_DELAYS",
        "AIRLINE_HIST_DELAY_RATE",
        "ARR_DEL15",
    )
    .orderBy(
        "OP_UNIQUE_CARRIER",
        "FL_DATE",
    )
    .limit(30)
)

### 3.2 Historical Origin-Airport Delay Rate

The `ORIGIN_HIST_DELAY_RATE` feature represents the prior arrival-delay rate of flights departing from each origin airport.

For every training record, only flights from earlier dates at the same origin airport are included. The current date and all future records are excluded to prevent target leakage.

A smoothed estimate is used, with the global training delay rate serving as the prior and as the fallback when no earlier airport history exists.

In [0]:
# Daily origin-airport delay statistics within the training period
origin_daily_stats = (
    df_train
    .groupBy(
        "ORIGIN",
        "FL_DATE",
    )
    .agg(
        F.count("*").alias("DAILY_ORIGIN_FLIGHT_COUNT"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "DAILY_ORIGIN_DELAY_COUNT"
        ),
    )
)

# Historical window excludes the current date
origin_history_window = (
    Window
    .partitionBy("ORIGIN")
    .orderBy(F.col("FL_DATE").cast("timestamp").cast("long"))
    .rowsBetween(Window.unboundedPreceding, -1)
)

origin_history_features = (
    origin_daily_stats
    .withColumn(
        "ORIGIN_PRIOR_FLIGHTS",
        F.sum("DAILY_ORIGIN_FLIGHT_COUNT").over(origin_history_window),
    )
    .withColumn(
        "ORIGIN_PRIOR_DELAYS",
        F.sum("DAILY_ORIGIN_DELAY_COUNT").over(origin_history_window),
    )
    .withColumn(
        "ORIGIN_HIST_DELAY_RATE",
        (
            F.coalesce(
                F.col("ORIGIN_PRIOR_DELAYS").cast("double"),
                F.lit(0.0),
            )
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.coalesce(
                F.col("ORIGIN_PRIOR_FLIGHTS").cast("double"),
                F.lit(0.0),
            )
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "ORIGIN",
        "FL_DATE",
        "ORIGIN_PRIOR_FLIGHTS",
        "ORIGIN_PRIOR_DELAYS",
        "ORIGIN_HIST_DELAY_RATE",
    )
)

# Join origin history onto the training dataset that already contains
# AIRLINE_HIST_DELAY_RATE
df_train_hist = (
    df_train_hist
    .join(
        origin_history_features,
        on=["ORIGIN", "FL_DATE"],
        how="left",
    )
    .withColumn(
        "ORIGIN_HIST_DELAY_RATE",
        F.coalesce(
            F.col("ORIGIN_HIST_DELAY_RATE"),
            F.lit(global_training_delay_rate),
        ),
    )
)

print(f"Training rows after origin join: {df_train_hist.count():,}")

display(
    df_train_hist
    .select(
        "ORIGIN",
        "FL_DATE",
        "ORIGIN_PRIOR_FLIGHTS",
        "ORIGIN_PRIOR_DELAYS",
        "ORIGIN_HIST_DELAY_RATE",
        "ARR_DEL15",
    )
    .orderBy(
        "ORIGIN",
        "FL_DATE",
    )
    .limit(30)
)

### 3.3 Historical Destination-Airport Delay Rate

The `DEST_HIST_DELAY_RATE` feature represents the prior arrival-delay rate of flights travelling to each destination airport.

For every training record, only flights from earlier dates with the same destination airport are included. The current date and all future observations are excluded to prevent target leakage.

A smoothed estimate is used, with the global training delay rate serving as the prior and as the fallback when no earlier destination history exists.

In [0]:
# Daily destination-airport delay statistics within the training period
dest_daily_stats = (
    df_train
    .groupBy(
        "DEST",
        "FL_DATE",
    )
    .agg(
        F.count("*").alias("DAILY_DEST_FLIGHT_COUNT"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "DAILY_DEST_DELAY_COUNT"
        ),
    )
)

# Historical window excludes the current date
dest_history_window = (
    Window
    .partitionBy("DEST")
    .orderBy(F.col("FL_DATE").cast("timestamp").cast("long"))
    .rowsBetween(Window.unboundedPreceding, -1)
)

dest_history_features = (
    dest_daily_stats
    .withColumn(
        "DEST_PRIOR_FLIGHTS",
        F.sum("DAILY_DEST_FLIGHT_COUNT").over(dest_history_window),
    )
    .withColumn(
        "DEST_PRIOR_DELAYS",
        F.sum("DAILY_DEST_DELAY_COUNT").over(dest_history_window),
    )
    .withColumn(
        "DEST_HIST_DELAY_RATE",
        (
            F.coalesce(
                F.col("DEST_PRIOR_DELAYS").cast("double"),
                F.lit(0.0),
            )
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.coalesce(
                F.col("DEST_PRIOR_FLIGHTS").cast("double"),
                F.lit(0.0),
            )
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "DEST",
        "FL_DATE",
        "DEST_PRIOR_FLIGHTS",
        "DEST_PRIOR_DELAYS",
        "DEST_HIST_DELAY_RATE",
    )
)

# Join destination history onto the training dataset
df_train_hist = (
    df_train_hist
    .join(
        dest_history_features,
        on=["DEST", "FL_DATE"],
        how="left",
    )
    .withColumn(
        "DEST_HIST_DELAY_RATE",
        F.coalesce(
            F.col("DEST_HIST_DELAY_RATE"),
            F.lit(global_training_delay_rate),
        ),
    )
)

print(f"Training rows after destination join: {df_train_hist.count():,}")

display(
    df_train_hist
    .select(
        "DEST",
        "FL_DATE",
        "DEST_PRIOR_FLIGHTS",
        "DEST_PRIOR_DELAYS",
        "DEST_HIST_DELAY_RATE",
        "ARR_DEL15",
    )
    .orderBy(
        "DEST",
        "FL_DATE",
    )
    .limit(30)
)

### 3.4 Historical Route Delay Rate

The `ROUTE_HIST_DELAY_RATE` feature represents the prior arrival-delay rate for each origin–destination route.

For every training record, only flights from earlier dates on the same route are included. The current date and all future observations are excluded to prevent target leakage.

Because some routes have limited historical volume, a smoothed estimate is used. The global training delay rate serves as the prior and as the fallback when no earlier route history exists.

In [0]:
# Daily route-level delay statistics within the training period
route_daily_stats = (
    df_train
    .groupBy(
        "ORIGIN",
        "DEST",
        "FL_DATE",
    )
    .agg(
        F.count("*").alias("DAILY_ROUTE_FLIGHT_COUNT"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "DAILY_ROUTE_DELAY_COUNT"
        ),
    )
)

# Historical window excludes the current date
route_history_window = (
    Window
    .partitionBy("ORIGIN", "DEST")
    .orderBy(F.col("FL_DATE").cast("timestamp").cast("long"))
    .rowsBetween(Window.unboundedPreceding, -1)
)

route_history_features = (
    route_daily_stats
    .withColumn(
        "ROUTE_PRIOR_FLIGHTS",
        F.sum("DAILY_ROUTE_FLIGHT_COUNT").over(route_history_window),
    )
    .withColumn(
        "ROUTE_PRIOR_DELAYS",
        F.sum("DAILY_ROUTE_DELAY_COUNT").over(route_history_window),
    )
    .withColumn(
        "ROUTE_HIST_DELAY_RATE",
        (
            F.coalesce(
                F.col("ROUTE_PRIOR_DELAYS").cast("double"),
                F.lit(0.0),
            )
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.coalesce(
                F.col("ROUTE_PRIOR_FLIGHTS").cast("double"),
                F.lit(0.0),
            )
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "ORIGIN",
        "DEST",
        "FL_DATE",
        "ROUTE_PRIOR_FLIGHTS",
        "ROUTE_PRIOR_DELAYS",
        "ROUTE_HIST_DELAY_RATE",
    )
)

# Join route history onto the training dataset
df_train_hist = (
    df_train_hist
    .join(
        route_history_features,
        on=["ORIGIN", "DEST", "FL_DATE"],
        how="left",
    )
    .withColumn(
        "ROUTE_HIST_DELAY_RATE",
        F.coalesce(
            F.col("ROUTE_HIST_DELAY_RATE"),
            F.lit(global_training_delay_rate),
        ),
    )
)

print(f"Training rows after route join: {df_train_hist.count():,}")

display(
    df_train_hist
    .select(
        "ORIGIN",
        "DEST",
        "FL_DATE",
        "ROUTE_PRIOR_FLIGHTS",
        "ROUTE_PRIOR_DELAYS",
        "ROUTE_HIST_DELAY_RATE",
        "ARR_DEL15",
    )
    .orderBy(
        "ORIGIN",
        "DEST",
        "FL_DATE",
    )
    .limit(30)
)

### 3.5 Apply Training-Only Historical Features to Validation and Test Data

Historical mappings for airlines, origin airports, destination airports, and routes are calculated exclusively from the training period.

These fixed training-period mappings are then joined to the validation and test datasets. Neither validation nor test outcomes are used when calculating the historical rates.

For categories not observed during training, the global training delay rate is used as a fallback.

In [0]:
# -------------------------------------------------------
# Create historical mappings from training data only
# -------------------------------------------------------

airline_training_map = (
    df_train
    .groupBy("OP_UNIQUE_CARRIER")
    .agg(
        F.count("*").alias("AIRLINE_TRAIN_FLIGHTS"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "AIRLINE_TRAIN_DELAYS"
        ),
    )
    .withColumn(
        "AIRLINE_HIST_DELAY_RATE",
        (
            F.col("AIRLINE_TRAIN_DELAYS").cast("double")
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.col("AIRLINE_TRAIN_FLIGHTS").cast("double")
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "OP_UNIQUE_CARRIER",
        "AIRLINE_HIST_DELAY_RATE",
    )
)

origin_training_map = (
    df_train
    .groupBy("ORIGIN")
    .agg(
        F.count("*").alias("ORIGIN_TRAIN_FLIGHTS"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "ORIGIN_TRAIN_DELAYS"
        ),
    )
    .withColumn(
        "ORIGIN_HIST_DELAY_RATE",
        (
            F.col("ORIGIN_TRAIN_DELAYS").cast("double")
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.col("ORIGIN_TRAIN_FLIGHTS").cast("double")
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "ORIGIN",
        "ORIGIN_HIST_DELAY_RATE",
    )
)

dest_training_map = (
    df_train
    .groupBy("DEST")
    .agg(
        F.count("*").alias("DEST_TRAIN_FLIGHTS"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "DEST_TRAIN_DELAYS"
        ),
    )
    .withColumn(
        "DEST_HIST_DELAY_RATE",
        (
            F.col("DEST_TRAIN_DELAYS").cast("double")
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.col("DEST_TRAIN_FLIGHTS").cast("double")
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "DEST",
        "DEST_HIST_DELAY_RATE",
    )
)

route_training_map = (
    df_train
    .groupBy("ORIGIN", "DEST")
    .agg(
        F.count("*").alias("ROUTE_TRAIN_FLIGHTS"),
        F.sum(F.col("ARR_DEL15").cast("long")).alias(
            "ROUTE_TRAIN_DELAYS"
        ),
    )
    .withColumn(
        "ROUTE_HIST_DELAY_RATE",
        (
            F.col("ROUTE_TRAIN_DELAYS").cast("double")
            + F.lit(SMOOTHING_STRENGTH * global_training_delay_rate)
        )
        /
        (
            F.col("ROUTE_TRAIN_FLIGHTS").cast("double")
            + F.lit(SMOOTHING_STRENGTH)
        ),
    )
    .select(
        "ORIGIN",
        "DEST",
        "ROUTE_HIST_DELAY_RATE",
    )
)


def attach_training_history(dataset: DataFrame) -> DataFrame:
    """Attach historical rates calculated exclusively from training data."""
    return (
        dataset
        .join(
            airline_training_map,
            on="OP_UNIQUE_CARRIER",
            how="left",
        )
        .join(
            origin_training_map,
            on="ORIGIN",
            how="left",
        )
        .join(
            dest_training_map,
            on="DEST",
            how="left",
        )
        .join(
            route_training_map,
            on=["ORIGIN", "DEST"],
            how="left",
        )
        .fillna(
            {
                "AIRLINE_HIST_DELAY_RATE": global_training_delay_rate,
                "ORIGIN_HIST_DELAY_RATE": global_training_delay_rate,
                "DEST_HIST_DELAY_RATE": global_training_delay_rate,
                "ROUTE_HIST_DELAY_RATE": global_training_delay_rate,
            }
        )
    )


df_validation_hist = attach_training_history(df_validation)
df_test_hist = attach_training_history(df_test)

print(
    f"Validation rows after historical joins: "
    f"{df_validation_hist.count():,}"
)
print(
    f"Test rows after historical joins: "
    f"{df_test_hist.count():,}"
)

display(
    df_validation_hist
    .select(
        "FL_DATE",
        "OP_UNIQUE_CARRIER",
        "ORIGIN",
        "DEST",
        "AIRLINE_HIST_DELAY_RATE",
        "ORIGIN_HIST_DELAY_RATE",
        "DEST_HIST_DELAY_RATE",
        "ROUTE_HIST_DELAY_RATE",
        "ARR_DEL15",
    )
    .limit(20)
)

### 3.6 Validate Historical Performance Features

The historical feature datasets are validated before categorical encoding and model training.

This check confirms that:

- Record counts remain unchanged after historical-feature joins
- No historical delay-rate features contain missing values
- All historical rates fall within the valid probability range of 0 to 1
- Training, validation, and test datasets contain the same historical feature columns

In [0]:
HISTORICAL_RATE_COLUMNS = [
    "AIRLINE_HIST_DELAY_RATE",
    "ORIGIN_HIST_DELAY_RATE",
    "DEST_HIST_DELAY_RATE",
    "ROUTE_HIST_DELAY_RATE",
]

historical_validation_rows = []

for dataset_name, dataset in [
    ("TRAIN", df_train_hist),
    ("VALIDATION", df_validation_hist),
    ("TEST", df_test_hist),
]:
    summary_row = (
        dataset
        .select(
            F.lit(dataset_name).alias("DATASET"),
            F.count("*").alias("TOTAL_RECORDS"),
            *[
                F.sum(
                    F.when(F.col(column_name).isNull(), 1).otherwise(0)
                ).alias(f"{column_name}_NULLS")
                for column_name in HISTORICAL_RATE_COLUMNS
            ],
            *[
                F.sum(
                    F.when(
                        (F.col(column_name) < 0)
                        | (F.col(column_name) > 1),
                        1,
                    ).otherwise(0)
                ).alias(f"{column_name}_OUT_OF_RANGE")
                for column_name in HISTORICAL_RATE_COLUMNS
            ],
        )
    )

    historical_validation_rows.append(summary_row)

historical_validation_summary = historical_validation_rows[0]

for summary_row in historical_validation_rows[1:]:
    historical_validation_summary = (
        historical_validation_summary.unionByName(summary_row)
    )

display(historical_validation_summary)

## 4. Encode Categorical Variables

Categorical and numerical predictors are converted into a model-ready sparse feature vector using Spark's `FeatureHasher`.

Feature hashing maps categorical values into a fixed-dimensional numerical representation without fitting and storing large category-indexing models. This approach avoids the Spark Connect ML model-cache limitation encountered with `StringIndexer` and `OneHotEncoder`.

The same deterministic transformation is applied to the training, validation, and test datasets. No validation or test outcomes are used during preprocessing.

Before applying the transformation, the preprocessing configuration is validated to ensure that all required model input columns are present across the chronological training, validation, and test datasets. The resulting hashed datasets retain only `FL_DATE`, the target variable, and the generated `features` vector required by the machine learning algorithms. Each transformed dataset is then validated to ensure that feature hashing has been applied successfully before model training begins.

In [0]:
from pyspark.ml.feature import FeatureHasher


# ============================================================
# 1. Define model input columns
# ============================================================

CATEGORICAL_COLUMNS = [
    "OP_UNIQUE_CARRIER",
    "ORIGIN",
    "DEST",
    "ORIGIN_CITY_NAME",
    "ORIGIN_STATE_NM",
    "DEST_CITY_NAME",
    "DEST_STATE_NM",
    "SEASON",
    "TIME_OF_DAY",
    "FLIGHT_DISTANCE_CATEGORY",
]

NUMERICAL_COLUMNS = [
    "QUARTER",
    "MONTH",
    "DAY_OF_WEEK",
    "DISTANCE",
    "CRS_DEP_TIME",
    "CRS_ARR_TIME",
    "CRS_ELAPSED_TIME",
    "DEP_HOUR",
    "DEP_MINUTE",
    "IS_WEEKEND",
    "AIRLINE_HIST_DELAY_RATE",
    "ORIGIN_HIST_DELAY_RATE",
    "DEST_HIST_DELAY_RATE",
    "ROUTE_HIST_DELAY_RATE",
]

MODEL_INPUT_COLUMNS = CATEGORICAL_COLUMNS + NUMERICAL_COLUMNS


# ============================================================
# 2. Validate required columns across all datasets
# ============================================================

required_columns = set(
    MODEL_INPUT_COLUMNS
    + [
        "FL_DATE",
        TARGET_COLUMN,
    ]
)

for dataframe_name, dataframe in {
    "df_train_hist": df_train_hist,
    "df_validation_hist": df_validation_hist,
    "df_test_hist": df_test_hist,
}.items():
    missing_columns = sorted(
        required_columns - set(dataframe.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{dataframe_name} preprocessing validation failed. "
            f"Missing required columns: {missing_columns}"
        )


# ============================================================
# 3. Configure FeatureHasher
# ============================================================

# Fixed-size sparse feature vector.
# Using a power of two provides an efficient hash space.
HASH_VECTOR_SIZE = 2 ** 12  # 4,096 positions

feature_hasher = FeatureHasher(
    inputCols=MODEL_INPUT_COLUMNS,
    outputCol="features",
    categoricalCols=CATEGORICAL_COLUMNS,
    numFeatures=HASH_VECTOR_SIZE,
)


# ============================================================
# 4. Apply feature hashing to train, validation, and test data
# ============================================================

df_train_hashed = (
    feature_hasher
    .transform(df_train_hist)
    .select(
        "FL_DATE",
        TARGET_COLUMN,
        "features",
    )
)

df_validation_hashed = (
    feature_hasher
    .transform(df_validation_hist)
    .select(
        "FL_DATE",
        TARGET_COLUMN,
        "features",
    )
)

df_test_hashed = (
    feature_hasher
    .transform(df_test_hist)
    .select(
        "FL_DATE",
        TARGET_COLUMN,
        "features",
    )
)


# ============================================================
# 5. Force lineage validation
# ============================================================

for dataframe_name, dataframe in {
    "df_train_hashed": df_train_hashed,
    "df_validation_hashed": df_validation_hashed,
    "df_test_hashed": df_test_hashed,
}.items():
    validation_row_count = (
        dataframe
        .select("features")
        .limit(1)
        .count()
    )

    if validation_row_count == 0:
        raise ValueError(
            f"{dataframe_name} contains no rows."
        )

    print(
        f"{dataframe_name} created and validated successfully."
    )


# ============================================================
# 6. Configuration summary
# ============================================================

print()
print("Feature-hashing preprocessing configured successfully.")
print(f"Categorical features: {len(CATEGORICAL_COLUMNS)}")
print(f"Numerical features: {len(NUMERICAL_COLUMNS)}")
print(f"Total raw predictors: {len(MODEL_INPUT_COLUMNS)}")
print(f"Hashed vector size: {HASH_VECTOR_SIZE:,}")
print(f"Target: {TARGET_COLUMN}")

### 4.1 Apply Feature Hashing to Model Datasets

The configured `FeatureHasher` is applied to the chronological training, validation, and test datasets.

This transformation converts the selected numerical and categorical predictors into a fixed-size sparse vector stored in the `features` column required by Spark machine learning estimators.

The transformed DataFrames retain `FL_DATE` and the target variable so they can be used for chronological hyperparameter tuning and final model evaluation.

In [0]:
df_train_hashed = feature_hasher.transform(df_train_hist)

df_validation_hashed = feature_hasher.transform(df_validation_hist)

df_test_hashed = feature_hasher.transform(df_test_hist)


required_model_columns = {
    "FL_DATE",
    TARGET_COLUMN,
    "features",
}

for dataframe_name, dataframe in {
    "df_train_hashed": df_train_hashed,
    "df_validation_hashed": df_validation_hashed,
    "df_test_hashed": df_test_hashed,
}.items():
    missing_columns = required_model_columns.difference(dataframe.columns)

    if missing_columns:
        raise ValueError(
            f"{dataframe_name} is missing required columns: "
            f"{sorted(missing_columns)}"
        )

    print(
        f"{dataframe_name} created successfully with "
        f"{dataframe.count():,} rows."
    )

### 4.2 Fit and Apply the Preprocessing Pipeline

The preprocessing pipeline is fitted exclusively on the training dataset.

The fitted pipeline is then applied unchanged to the training, validation, and test datasets. This ensures that category indexing and one-hot encoding are learned only from the training period.

The resulting `features` column contains the assembled numerical and encoded categorical predictors required by Spark ML models.

In [0]:
df_train_prepared = feature_hasher.transform(df_train_hist)
df_validation_prepared = feature_hasher.transform(df_validation_hist)
df_test_prepared = feature_hasher.transform(df_test_hist)

print("Feature hashing applied successfully.")
print(f"Training rows: {df_train_prepared.count():,}")
print(f"Validation rows: {df_validation_prepared.count():,}")
print(f"Test rows: {df_test_prepared.count():,}")

display(
    df_train_prepared
    .select(
        "FL_DATE",
        "OP_UNIQUE_CARRIER",
        "ORIGIN",
        "DEST",
        "features",
        "ARR_DEL15",
    )
    .limit(10)
)

## 5. Train Multiple Candidate Models

Multiple classification algorithms will be trained and compared using the same chronologically separated datasets.

The first candidate is Logistic Regression, which serves as the baseline model. It provides a computationally efficient benchmark and estimates the probability that a scheduled flight will arrive at least 15 minutes late.

The model is trained using the training period only. Validation data will be used later to evaluate performance and guide model selection.

### 5.1 Majority Class Baseline

Before training machine learning models, a simple majority-class baseline is evaluated.

The majority-class classifier predicts every flight as the most frequent class observed in the training data. In this dataset, the majority class is **on-time arrival (ARR_DEL15 = 0)**.

Although this baseline is intentionally simple, it provides an important reference point for determining whether more sophisticated machine learning models deliver meaningful predictive improvements.

In [0]:
from pyspark.sql import functions as F
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)

# ---------------------------------------------------
# Majority class prediction
# ---------------------------------------------------

majority_class = (
    df_train_prepared
    .groupBy(TARGET_COLUMN)
    .count()
    .orderBy(F.desc("count"))
    .first()[TARGET_COLUMN]
)

baseline_predictions = (
    df_validation_prepared
    .withColumn(
        "prediction",
        F.lit(float(majority_class))
    )
    .withColumn(
        "rawPrediction",
        F.array(
            F.lit(1.0),
            F.lit(0.0)
        )
    )
    .withColumn(
        "probability",
        F.array(
            F.lit(1.0),
            F.lit(0.0)
        )
    )
)

print(f"Majority class: {majority_class}")

#### Majority-Class Baseline Result

The training dataset's majority class is `ARR_DEL15 = 0`, representing flights that arrived less than 15 minutes late.

Therefore, the majority-class baseline predicts every validation record as on time. This provides a simple lower-bound benchmark for evaluating whether the machine-learning models produce meaningful improvement.

### 5.1.1 Evaluate the Majority-Class Baseline

The majority-class baseline is evaluated on the validation dataset using the same performance metrics that will be applied to all candidate machine learning models.

Although this classifier predicts every flight as the majority class (on-time arrival), it provides an important benchmark for determining whether more sophisticated models deliver meaningful predictive improvements.

The following metrics are reported:

- Accuracy
- Weighted Precision
- Weighted Recall
- Weighted F1-score

ROC AUC and PR AUC are not applicable because the majority-class baseline does not produce meaningful probability estimates.

In [0]:
baseline_accuracy = MulticlassClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    metricName="accuracy",
).evaluate(baseline_predictions)

baseline_precision = MulticlassClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    metricName="weightedPrecision",
).evaluate(baseline_predictions)

baseline_recall = MulticlassClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    metricName="weightedRecall",
).evaluate(baseline_predictions)

baseline_f1 = MulticlassClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    metricName="f1",
).evaluate(baseline_predictions)

majority_baseline_results = spark.createDataFrame(
    [
        (
            "Majority Class Baseline",
            round(baseline_accuracy, 4),
            round(baseline_precision, 4),
            round(baseline_recall, 4),
            round(baseline_f1, 4),
        )
    ],
    [
        "MODEL",
        "ACCURACY",
        "WEIGHTED_PRECISION",
        "WEIGHTED_RECALL",
        "WEIGHTED_F1_SCORE",
    ],
)

display(majority_baseline_results)

### Majority-Class Baseline Interpretation

The majority-class baseline achieved an overall accuracy of **81.45%**, slightly exceeding the Logistic Regression baseline.

However, this classifier predicts every flight as the majority class (on-time arrival) and therefore cannot identify delayed flights. Consequently, the baseline provides a useful lower-bound benchmark but is not suitable for operational decision support.

The Logistic Regression model is expected to outperform the majority-class baseline on delay-specific evaluation metrics despite having a similar overall accuracy.

### 5.1.2 Majority-Class Baseline Confusion Matrix

A confusion matrix is generated to evaluate the prediction behavior of the majority-class baseline.

Since this classifier predicts every flight as the majority class (on-time arrival), the confusion matrix illustrates its inability to identify delayed flights despite achieving relatively high overall accuracy.

The confusion matrix reports:

- True Negatives (TN)
- False Positives (FP)
- False Negatives (FN)
- True Positives (TP)

This analysis provides a reference point for comparing more sophisticated machine learning models.

In [0]:
baseline_confusion = (
    baseline_predictions
    .groupBy(
        F.col(TARGET_COLUMN).alias("ACTUAL"),
        F.col("prediction").cast("int").alias("PREDICTED"),
    )
    .count()
    .orderBy(
        "ACTUAL",
        "PREDICTED",
    )
)

display(baseline_confusion)

#### Majority-Class Baseline Confusion Matrix Interpretation

The majority-class baseline predicts every validation observation as the majority class (`ARR_DEL15 = 0`).

Consequently:

- All on-time flights are predicted correctly.
- No delayed flights are identified.
- The model produces no false-positive predictions because it never predicts the delayed class.
- Every delayed flight becomes a false negative.

Although this classifier achieves relatively high overall accuracy due to the class imbalance, it has no operational value because it cannot identify flights at risk of delay.

### 5.2 Logistic Regression Baseline

A binary Logistic Regression model is trained using the assembled `features` vector and the `ARR_DEL15` target.

The initial model uses moderate regularization and a limited number of iterations to establish a baseline. Hyperparameter tuning will be performed later after the candidate models have been compared.

In [0]:
from pyspark.ml.classification import LogisticRegression


logistic_regression = LogisticRegression(
    featuresCol="features",
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=20,
    regParam=0.01,
    elasticNetParam=0.0,
    standardization=True,
    family="binomial",
)

logistic_model = logistic_regression.fit(
    df_train_prepared.select(
        "features",
        TARGET_COLUMN,
    )
)

print("Logistic Regression baseline trained successfully.")
print(f"Iterations completed: {logistic_model.summary.totalIterations}")
print(f"Intercept: {logistic_model.intercept:.6f}")
print(f"Coefficient vector size: {logistic_model.coefficients.size}")

### 5.3 Evaluate the Logistic Regression Baseline

The Logistic Regression model is first evaluated on the validation dataset before additional candidate models are trained.

The validation dataset was not used during model fitting, making it suitable for an unbiased assessment of predictive performance.

The following performance metrics are reported:

- Accuracy
- Precision
- Recall
- F1-score
- ROC Area Under the Curve (ROC AUC)
- Precision–Recall Area Under the Curve (PR AUC)

These metrics establish the baseline against which subsequent models will be compared.

In [0]:
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)

# ----------------------------------------------------
# Generate validation predictions
# ----------------------------------------------------

validation_predictions = logistic_model.transform(
    df_validation_prepared
)

# ----------------------------------------------------
# Classification metrics
# ----------------------------------------------------

accuracy = MulticlassClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    metricName="accuracy",
).evaluate(validation_predictions)

precision = MulticlassClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    metricName="weightedPrecision",
).evaluate(validation_predictions)

recall = MulticlassClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    metricName="weightedRecall",
).evaluate(validation_predictions)

f1_score = MulticlassClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    metricName="f1",
).evaluate(validation_predictions)

roc_auc = BinaryClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
).evaluate(validation_predictions)

pr_auc = BinaryClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR",
).evaluate(validation_predictions)

# ----------------------------------------------------
# Display results
# ----------------------------------------------------

baseline_results = spark.createDataFrame(
    [
        (
            "Logistic Regression",
            round(accuracy, 4),
            round(precision, 4),
            round(recall, 4),
            round(f1_score, 4),
            round(roc_auc, 4),
            round(pr_auc, 4),
        )
    ],
    [
        "MODEL",
        "ACCURACY",
        "PRECISION",
        "RECALL",
        "F1_SCORE",
        "ROC_AUC",
        "PR_AUC",
    ],
)

display(baseline_results)

### 5.4 Logistic Regression Confusion Matrix

A confusion matrix is generated to examine the classification outcomes of the Logistic Regression baseline.

The confusion matrix summarizes:

- True Negatives (TN)
- False Positives (FP)
- False Negatives (FN)
- True Positives (TP)

This provides additional insight into the types of prediction errors made by the model beyond the aggregate evaluation metrics.

In [0]:
confusion_matrix = (
    validation_predictions
    .groupBy(
        F.col(TARGET_COLUMN).alias("ACTUAL"),
        F.col("prediction").cast("int").alias("PREDICTED"),
    )
    .count()
    .orderBy(
        "ACTUAL",
        "PREDICTED",
    )
)

display(confusion_matrix)

### Confusion Matrix Interpretation

The Logistic Regression confusion matrix is interpreted as follows:

- **True Negatives (TN):** Correctly predicted on-time flights.
- **False Positives (FP):** Flights predicted as delayed that actually arrived on time.
- **False Negatives (FN):** Flights predicted as on time that were actually delayed.
- **True Positives (TP):** Correctly predicted delayed flights.

This analysis provides insight into the model's prediction errors beyond overall accuracy.

In [0]:
TN = confusion_matrix.filter(
    (F.col("ACTUAL") == 0) &
    (F.col("PREDICTED") == 0)
).first()["count"]

FP = confusion_matrix.filter(
    (F.col("ACTUAL") == 0) &
    (F.col("PREDICTED") == 1)
).first()["count"]

FN = confusion_matrix.filter(
    (F.col("ACTUAL") == 1) &
    (F.col("PREDICTED") == 0)
).first()["count"]

TP = confusion_matrix.filter(
    (F.col("ACTUAL") == 1) &
    (F.col("PREDICTED") == 1)
).first()["count"]

confusion_summary = spark.createDataFrame(
    [
        ("True Negative (TN)", TN),
        ("False Positive (FP)", FP),
        ("False Negative (FN)", FN),
        ("True Positive (TP)", TP),
    ],
    ["CLASSIFICATION_RESULT", "COUNT"],
)

display(confusion_summary)

### Logistic Regression Confusion-Matrix Interpretation

The Logistic Regression baseline correctly classified most on-time flights, producing 939,046 true negatives and only 5,742 false positives.

However, the model detected only 3,382 delayed flights while incorrectly classifying 211,728 delayed flights as on time. This corresponds to a delayed-flight recall of approximately 1.57%.

Therefore, the baseline's high accuracy is largely driven by strong performance on the majority on-time class. Additional candidate models, class weighting, and decision-threshold tuning are necessary to improve delayed-flight detection.

### Model Performance Comparison

To support objective model selection, the performance of every candidate model is summarized in a single comparison table.

Each model is evaluated using the same validation dataset and the same evaluation metrics.

The comparison includes:

- Accuracy
- Precision
- Recall
- F1-score
- ROC AUC
- Precision–Recall AUC

This table will be updated as additional candidate models are trained and evaluated.

In [0]:
model_comparison = spark.createDataFrame(
    [
        (
            "Majority Class Baseline",
            baseline_accuracy,
            baseline_precision,
            baseline_recall,
            baseline_f1,
            None,
            None,
        ),
        (
            "Logistic Regression",
            accuracy,
            precision,
            recall,
            f1_score,
            roc_auc,
            pr_auc,
        ),
    ],
    schema="""
        MODEL string,
        ACCURACY double,
        PRECISION double,
        RECALL double,
        F1_SCORE double,
        ROC_AUC double,
        PR_AUC double
    """,
)

display(
    model_comparison
    .orderBy("MODEL")
)

#### Comparison-Table Note

ROC AUC and Precision–Recall AUC are not reported for the majority-class baseline because it assigns the same prediction to every record and does not generate meaningful risk rankings.

The null values are therefore expected and do not indicate a data-processing error.

### 5.5 Random Forest Classifier

The second candidate model is a Random Forest classifier.

Random Forest can capture nonlinear relationships and interactions among predictors that may not be represented adequately by Logistic Regression.

Because ensemble-tree training on the complete 4.59-million-record dataset is computationally expensive in the available Serverless environment, the model is trained using a reproducible stratified sample of the training period. Stratified sampling preserves both on-time and delayed-flight observations while maintaining training-only data use.

The fitted model will still be evaluated on the complete validation dataset.

In [0]:
# -------------------------------------------------------
# Reproducible stratified sample for tree-based models
# -------------------------------------------------------

TREE_SAMPLE_FRACTIONS = {
    0: 0.10,
    1: 0.30,
}

df_train_tree = (
    df_train_prepared
    .sampleBy(
        col=TARGET_COLUMN,
        fractions=TREE_SAMPLE_FRACTIONS,
        seed=42,
    )
    .select(
        "features",
        TARGET_COLUMN,
    )
)

tree_sample_summary = (
    df_train_tree
    .groupBy(TARGET_COLUMN)
    .count()
    .orderBy(TARGET_COLUMN)
)

tree_sample_count = df_train_tree.count()

print(f"Tree-model training sample rows: {tree_sample_count:,}")
display(tree_sample_summary)

#### Stratified Training Sample Interpretation

The stratified sampling procedure retained approximately 670 thousand training records while increasing the representation of delayed flights.

Compared with the original training dataset, the sampled dataset contains a substantially more balanced class distribution, allowing tree-based models to learn delay-related patterns more effectively while reducing computational cost.

Only the training dataset was sampled. The validation and test datasets remain unchanged to preserve an unbiased evaluation of model performance.

In [0]:
from pyspark.ml.classification import RandomForestClassifier

random_forest = RandomForestClassifier(
    featuresCol="features",
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",

    # Conservative baseline parameters
    numTrees=5,
    maxDepth=6,
    maxBins=32,
    minInstancesPerNode=20,

    seed=42,
)

random_forest_model = random_forest.fit(
    df_train_prepared.select(
        "features",
        TARGET_COLUMN,
    )
)

print("Random Forest trained successfully.")
print(f"Trees: {random_forest_model.getNumTrees}")

### 5.6 Evaluate the Random Forest Baseline

The Random Forest classifier is evaluated using the validation dataset.

The validation dataset was not used during model training and therefore provides an unbiased estimate of predictive performance.

The following evaluation metrics are reported:

- Accuracy
- Precision
- Recall
- F1-score
- ROC Area Under the Curve (ROC AUC)
- Precision–Recall Area Under the Curve (PR AUC)

These metrics are compared directly with the Majority-Class Baseline and Logistic Regression models.

In [0]:
rf_validation_predictions = random_forest_model.transform(
    df_validation_prepared
)

rf_accuracy = MulticlassClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    metricName="accuracy",
).evaluate(rf_validation_predictions)

rf_precision = MulticlassClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    metricName="weightedPrecision",
).evaluate(rf_validation_predictions)

rf_recall = MulticlassClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    metricName="weightedRecall",
).evaluate(rf_validation_predictions)

rf_f1 = MulticlassClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    metricName="f1",
).evaluate(rf_validation_predictions)

rf_roc_auc = BinaryClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
).evaluate(rf_validation_predictions)

rf_pr_auc = BinaryClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR",
).evaluate(rf_validation_predictions)

rf_results = spark.createDataFrame(
    [
        (
            "Random Forest",
            round(rf_accuracy, 4),
            round(rf_precision, 4),
            round(rf_recall, 4),
            round(rf_f1, 4),
            round(rf_roc_auc, 4),
            round(rf_pr_auc, 4),
        )
    ],
    [
        "MODEL",
        "ACCURACY",
        "PRECISION",
        "RECALL",
        "F1_SCORE",
        "ROC_AUC",
        "PR_AUC",
    ],
)

display(rf_results)

### 5.7 Random Forest Confusion Matrix

A confusion matrix is generated to evaluate the classification behavior of the baseline Random Forest model.

The confusion matrix summarizes:

- True Negatives (TN)
- False Positives (FP)
- False Negatives (FN)
- True Positives (TP)

These results provide insight into the types of prediction errors made by the Random Forest model and serve as the baseline before hyperparameter tuning.

In [0]:
rf_confusion_matrix = (
    rf_validation_predictions
    .groupBy(
        F.col(TARGET_COLUMN).alias("ACTUAL"),
        F.col("prediction").cast("int").alias("PREDICTED"),
    )
    .count()
    .orderBy(
        "ACTUAL",
        "PREDICTED",
    )
)

display(rf_confusion_matrix)

#### Random Forest Baseline Interpretation

The baseline Random Forest classifier predicted every validation observation as the majority class (on-time arrival), producing the same confusion matrix as the Majority-Class Baseline.

Although the model completed training successfully, the selected baseline hyperparameters were too conservative to identify delayed flights.

This outcome establishes a meaningful baseline and motivates the hyperparameter-tuning stage, where model complexity will be increased to improve delayed-flight detection while maintaining generalization performance.

### Updated Model Performance Comparison

The comparison table is updated after completing the baseline models.

At this stage, three reference models have been evaluated:

- Majority-Class Baseline
- Logistic Regression
- Baseline Random Forest

The Random Forest results presented below correspond to the initial baseline configuration prior to hyperparameter tuning. The comparison table will be updated again after tuning to reflect the best-performing Random Forest model.

In [0]:
model_comparison = spark.createDataFrame(
    [
        (
            "Majority Class Baseline",
            round(baseline_accuracy, 4),
            round(baseline_precision, 4),
            round(baseline_recall, 4),
            round(baseline_f1, 4),
            None,
            None,
        ),
        (
            "Logistic Regression",
            round(accuracy, 4),
            round(precision, 4),
            round(recall, 4),
            round(f1_score, 4),
            round(roc_auc, 4),
            round(pr_auc, 4),
        ),
        (
            "Random Forest (Baseline)",
            round(rf_accuracy, 4),
            round(rf_precision, 4),
            round(rf_recall, 4),
            round(rf_f1, 4),
            round(rf_roc_auc, 4),
            round(rf_pr_auc, 4),
        ),
    ],
    schema="""
        MODEL string,
        ACCURACY double,
        PRECISION double,
        RECALL double,
        F1_SCORE double,
        ROC_AUC double,
        PR_AUC double
    """
)

display(
    model_comparison.orderBy("MODEL")
)

#### Comparison Interpretation

The comparison of the baseline models demonstrates that overall accuracy alone is not an appropriate metric for evaluating flight-delay prediction because the dataset is dominated by on-time flights.

The Majority-Class Baseline achieved the highest overall accuracy by predicting every flight as on time, but it was unable to identify delayed flights and therefore provides little operational value.

The Logistic Regression model produced a slightly lower overall accuracy while achieving the highest ROC AUC, Precision–Recall AUC, and F1-score among the evaluated baseline models. These results indicate that Logistic Regression provides better discrimination between delayed and on-time flights despite the class imbalance.

The baseline Random Forest model achieved performance similar to the Majority-Class Baseline, suggesting that the initial hyperparameter configuration was too conservative and unable to learn meaningful delay patterns.

These results justify the next phase of the workflow, where Random Forest hyperparameters will be optimized to improve delayed-flight detection before comparing the model with a gradient-boosting algorithm.

### 5.8 Gradient Boosting Model Selection

The project requires one gradient-boosting algorithm for comparison with the baseline models.

The notebook first checks whether Spark-compatible XGBoost is available in the current Databricks environment.

Because the required package is not installed, the analysis proceeds using Spark ML's built-in **Gradient-Boosted Trees (GBTClassifier)**. This model satisfies the project requirement while remaining fully compatible with the available execution environment.

In [0]:
try:
    from xgboost.spark import SparkXGBClassifier

    XGBOOST_AVAILABLE = True
    print("Spark XGBoost is available.")
    print(f"Estimator: {SparkXGBClassifier.__name__}")

except Exception as error:
    XGBOOST_AVAILABLE = False
    print("Spark XGBoost is not available.")
    print(f"Error type: {type(error).__name__}")
    print(f"Error message: {error}")

### 5.9 Train the Gradient-Boosted Trees Baseline

A Gradient-Boosted Trees classifier is trained as the final baseline candidate model.

The model is trained using the same reproducible stratified training sample used for Random Forest. Conservative settings are used initially to maintain computational feasibility in the Databricks Serverless environment.

The baseline configuration will be evaluated on the complete validation dataset before hyperparameter tuning.

In [0]:
from pyspark.ml.classification import GBTClassifier


gradient_boosted_trees = GBTClassifier(
    featuresCol="features",
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",

    # Lightweight baseline
    maxIter=10,
    maxDepth=4,
    maxBins=32,
    minInstancesPerNode=20,
    stepSize=0.10,

    seed=42,
)

gradient_boosted_trees_model = gradient_boosted_trees.fit(
    df_train_tree.select(
        "features",
        TARGET_COLUMN,
    )
)

print("Gradient-Boosted Trees baseline trained successfully.")
print(f"Iterations: {gradient_boosted_trees.getMaxIter()}")
print(f"Maximum depth: {gradient_boosted_trees.getMaxDepth()}")
print(f"Learning rate: {gradient_boosted_trees.getStepSize()}")

### 5.10 Evaluate the Gradient-Boosted Trees Baseline

The Gradient-Boosted Trees classifier is evaluated using the complete validation dataset.

The validation dataset was not used during model training and therefore provides an unbiased estimate of predictive performance.

The following evaluation metrics are reported:

- Accuracy
- Precision
- Recall
- F1-score
- ROC Area Under the Curve (ROC AUC)
- Precision–Recall Area Under the Curve (PR AUC)

The results will be compared directly with the Majority-Class Baseline, Logistic Regression, and Random Forest models.

In [0]:
gbt_validation_predictions = gradient_boosted_trees_model.transform(
    df_validation_prepared
)

gbt_accuracy = MulticlassClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    metricName="accuracy",
).evaluate(gbt_validation_predictions)

gbt_precision = MulticlassClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    metricName="weightedPrecision",
).evaluate(gbt_validation_predictions)

gbt_recall = MulticlassClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    metricName="weightedRecall",
).evaluate(gbt_validation_predictions)

gbt_f1 = MulticlassClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    metricName="f1",
).evaluate(gbt_validation_predictions)

gbt_roc_auc = BinaryClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
).evaluate(gbt_validation_predictions)

gbt_pr_auc = BinaryClassificationEvaluator(
    labelCol=TARGET_COLUMN,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR",
).evaluate(gbt_validation_predictions)

gbt_results = spark.createDataFrame(
    [
        (
            "Gradient-Boosted Trees",
            round(gbt_accuracy, 4),
            round(gbt_precision, 4),
            round(gbt_recall, 4),
            round(gbt_f1, 4),
            round(gbt_roc_auc, 4),
            round(gbt_pr_auc, 4),
        )
    ],
    [
        "MODEL",
        "ACCURACY",
        "PRECISION",
        "RECALL",
        "F1_SCORE",
        "ROC_AUC",
        "PR_AUC",
    ],
)

display(gbt_results)

### 5.11 Gradient-Boosted Trees Confusion Matrix

A confusion matrix is generated to evaluate the classification behavior of the baseline Gradient-Boosted Trees model.

The confusion matrix summarizes:

- True Negatives (TN)
- False Positives (FP)
- False Negatives (FN)
- True Positives (TP)

These results provide additional insight into the prediction behavior of the Gradient-Boosted Trees model before hyperparameter tuning.

In [0]:
gbt_confusion_matrix = (
    gbt_validation_predictions
    .groupBy(
        F.col(TARGET_COLUMN).alias("ACTUAL"),
        F.col("prediction").cast("int").alias("PREDICTED"),
    )
    .count()
    .orderBy(
        "ACTUAL",
        "PREDICTED",
    )
)

display(gbt_confusion_matrix)

#### Gradient-Boosted Trees Confusion Matrix Interpretation

The baseline Gradient-Boosted Trees model produced a substantially different prediction pattern compared with the Majority-Class Baseline, Logistic Regression, and the baseline Random Forest model.

The confusion matrix shows that the model correctly identified **126,337 delayed flights (True Positives)** while correctly classifying **600,723 on-time flights (True Negatives)**. Compared with the previous baseline models, the Gradient-Boosted Trees classifier detected considerably more delayed flights, reducing the number of false negatives.

This improvement in delayed-flight detection was achieved at the expense of generating more false-positive predictions, resulting in a lower overall accuracy. However, because the primary objective of this project is to identify flights at risk of delay rather than simply maximize overall accuracy, this trade-off is acceptable.

The corresponding improvement in ROC AUC and Precision–Recall AUC further indicates that the Gradient-Boosted Trees model provides stronger discrimination between delayed and on-time flights than the previous baseline models. These results suggest that Gradient-Boosted Trees is a promising candidate for hyperparameter tuning and final model selection.

### Updated Baseline Model Comparison

The comparison table is updated after evaluating all required baseline models.

At this stage, the following candidate models have been completed:

- Majority-Class Baseline
- Logistic Regression
- Random Forest (Baseline)
- Gradient-Boosted Trees (Baseline)

The tree-based models shown below correspond to their initial baseline configurations prior to hyperparameter tuning. The comparison table will be updated after tuning to reflect the best-performing models.

In [0]:
model_comparison = spark.createDataFrame(
    [
        (
            "Majority Class Baseline",
            round(baseline_accuracy, 4),
            round(baseline_precision, 4),
            round(baseline_recall, 4),
            round(baseline_f1, 4),
            None,
            None,
        ),
        (
            "Logistic Regression",
            round(accuracy, 4),
            round(precision, 4),
            round(recall, 4),
            round(f1_score, 4),
            round(roc_auc, 4),
            round(pr_auc, 4),
        ),
        (
            "Random Forest (Baseline)",
            round(rf_accuracy, 4),
            round(rf_precision, 4),
            round(rf_recall, 4),
            round(rf_f1, 4),
            round(rf_roc_auc, 4),
            round(rf_pr_auc, 4),
        ),
        (
            "Gradient-Boosted Trees (Baseline)",
            round(gbt_accuracy, 4),
            round(gbt_precision, 4),
            round(gbt_recall, 4),
            round(gbt_f1, 4),
            round(gbt_roc_auc, 4),
            round(gbt_pr_auc, 4),
        ),
    ],
    schema="""
        MODEL string,
        ACCURACY double,
        PRECISION double,
        RECALL double,
        F1_SCORE double,
        ROC_AUC double,
        PR_AUC double
    """
)

display(model_comparison.orderBy("MODEL"))

#### Baseline Model Comparison Interpretation

The completed baseline comparison confirms that overall accuracy is not sufficient for evaluating flight-delay prediction.

The Majority-Class Baseline and baseline Random Forest achieved the highest accuracy because both effectively predicted every flight as on time. However, neither model identified delayed flights, making them unsuitable for operational prioritization.

Logistic Regression achieved the highest weighted F1-score and maintained strong ROC AUC and Precision–Recall AUC values. It therefore provides the strongest overall balance across the majority and minority classes among the baseline models.

Gradient-Boosted Trees achieved the highest ROC AUC and Precision–Recall AUC and identified substantially more delayed flights than the other models. Its lower accuracy reflects a larger number of false-positive delay alerts rather than an inability to detect the target class.

Because delayed-flight detection is the primary objective, Gradient-Boosted Trees represents the most promising baseline for further optimization. Logistic Regression should also remain an important comparison model because of its interpretability, computational efficiency, and balanced overall performance.

The next phase performs hyperparameter tuning within the training period before final validation-set model selection.

## 6. Hyperparameter Tuning

The baseline models established the initial predictive performance of each candidate algorithm. This phase improves those models by evaluating multiple hyperparameter configurations while preventing temporal data leakage.

Hyperparameter tuning is performed exclusively within the original January–August 2025 training period using chronological validation folds. Candidate configurations are compared using validation performance before the best-performing model is evaluated on the project's official validation dataset.

The Majority-Class Baseline is excluded because it contains no trainable parameters.

The tuning process consists of:

1. Establishing a chronological tuning strategy.
2. Creating reusable tuning utilities.
3. Tuning Logistic Regression.
4. Tuning Random Forest.
5. Tuning Gradient-Boosted Trees.
6. Comparing the tuned models.
7. Selecting the final model for testing and explainability.

### 6.1 Hyperparameter Tuning Strategy

Chronological validation is used to ensure that each candidate model is evaluated on future observations that were not available during training.

The January–August 2025 training period is divided into expanding-window validation folds. Earlier months are used for model training, while the immediately following month is used for validation.

Each hyperparameter configuration is evaluated using the same chronological folds.

The evaluation emphasizes metrics that reflect delayed-flight detection rather than overall classification accuracy, including:

- Delayed-flight Precision
- Delayed-flight Recall
- Delayed-flight F1-score
- ROC Area Under the Curve (ROC AUC)
- Precision–Recall Area Under the Curve (PR AUC)

The configuration achieving the best overall validation performance is selected for each algorithm.

#### 6.3 Logistic Regression Hyperparameter Tuning

Logistic Regression is first optimized by evaluating different combinations of regularization strength and elastic-net mixing.

These hyperparameters control model complexity and help reduce overfitting while maintaining good generalization performance.

The evaluated parameters include:

- Regularization parameter (`regParam`)
- Elastic-net mixing parameter (`elasticNetParam`)

### 6.2 Reusable Chronological Tuning Utilities

To ensure consistency across all candidate models, reusable helper functions are created for chronological hyperparameter tuning.

These utilities perform the following tasks:

1. Train a candidate model using the chronological training fold.
2. Generate predictions for the corresponding validation fold.
3. Compute delayed-flight Precision, Recall, and F1-score.
4. Compute ROC AUC and Precision–Recall AUC.
5. Average the fold-level metrics across all chronological validation folds.

The same utilities will be reused for Logistic Regression, Random Forest, and Gradient-Boosted Trees, ensuring that every model is evaluated using an identical methodology.

In [0]:
# -------------------------------------------------------
# Recreate the existing stratified sample with FL_DATE
# retained for chronological hyperparameter tuning
# -------------------------------------------------------

TREE_SAMPLE_FRACTIONS = {
    0: 0.10,
    1: 0.30,
}

df_train_tree_tuning = (
    df_train_prepared
    .sampleBy(
        col=TARGET_COLUMN,
        fractions=TREE_SAMPLE_FRACTIONS,
        seed=42,
    )
    .select(
        "FL_DATE",
        "features",
        TARGET_COLUMN,
    )
)

tree_tuning_profile = (
    df_train_tree_tuning
    .select(
        F.min("FL_DATE").alias("MIN_DATE"),
        F.max("FL_DATE").alias("MAX_DATE"),
        F.count("*").alias("TOTAL_RECORDS"),
        F.sum(
            F.when(F.col(TARGET_COLUMN) == 0, 1).otherwise(0)
        ).alias("ON_TIME_RECORDS"),
        F.sum(
            F.when(F.col(TARGET_COLUMN) == 1, 1).otherwise(0)
        ).alias("DELAYED_RECORDS"),
    )
)

display(tree_tuning_profile)

### Required Libraries

The required Python libraries for chronological hyperparameter tuning are imported in this section.

These libraries provide functionality for:

- Measuring computational training time.
- Creating reusable helper functions.
- Computing delayed-flight classification metrics.
- Computing ranking-based metrics such as ROC AUC and Precision–Recall AUC.

The imported modules will be reused throughout the hyperparameter tuning process for Logistic Regression, Random Forest, and Gradient-Boosted Trees.

### 6.4 Build the Chronological Hyperparameter-Tuning Framework

This section creates a self-contained and reusable framework for chronological hyperparameter tuning across Logistic Regression, Random Forest, and Gradient-Boosted Trees.

The framework first identifies the prepared training DataFrame containing the flight date, target variable, and model-ready feature vector. It then creates a stratified tuning dataset by sampling 10% of on-time flights and 30% of delayed flights. This sampling strategy reduces computational cost while increasing the representation of delayed flights during model tuning.

The sampled dataset is persisted as a Delta table so that the tuning workflow remains reproducible and can be rerun after a Databricks session restart.

Four expanding-window chronological validation folds are defined:

| Fold | Training period | Validation period |
|---|---|---|
| Fold 1 | January–April 2025 | May 2025 |
| Fold 2 | January–May 2025 | June 2025 |
| Fold 3 | January–June 2025 | July 2025 |
| Fold 4 | January–July 2025 | August 2025 |

For every candidate hyperparameter configuration, the framework:

1. Trains the model separately on each chronological training window.
2. Evaluates the model on the corresponding future validation month.
3. Calculates overall classification metrics:
   - Accuracy
   - Weighted Precision
   - Weighted Recall
   - Weighted F1-score
4. Calculates delayed-flight positive-class metrics:
   - Delayed-flight Precision
   - Delayed-flight Recall
   - Delayed-flight F1-score
5. Calculates ROC AUC and Precision–Recall AUC.
6. Records model-training time.
7. Averages all fold-level metrics into one comparison row.
8. Ranks candidate configurations primarily by delayed-flight Recall, followed by PR AUC, delayed-flight F1-score, and ROC AUC.

This design preserves temporal ordering, prevents future observations from being used to predict earlier periods, and ensures that all candidate models are evaluated using the same reproducible methodology.

In [0]:
from __future__ import annotations

import time
from typing import Callable

from pyspark.sql import functions as F

from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)

In [0]:
from __future__ import annotations

import time
from collections.abc import Callable

from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


# ============================================================
# 1. Global tuning configuration
# ============================================================

TARGET_COLUMN = globals().get("TARGET_COLUMN", "ARR_DEL15")
RANDOM_SEED = 42

REQUIRED_TUNING_COLUMNS = {
    "FL_DATE",
    TARGET_COLUMN,
    "features",
}


# ============================================================
# 2. Locate the existing prepared training DataFrame
# ============================================================

# The first existing DataFrame containing FL_DATE, features,
# and the target column will be used as the tuning source.
#
# Add another variable name here only if your prepared training
# DataFrame uses a different name.

SOURCE_DATAFRAME_CANDIDATES = [
    "df_train_encoded",
    "df_train_hashed",
    "df_train_features",
    "df_train_model",
    "df_train",
]

source_training_df = None
source_training_df_name = None

for candidate_name in SOURCE_DATAFRAME_CANDIDATES:
    candidate_object = globals().get(candidate_name)

    if isinstance(candidate_object, DataFrame):
        candidate_columns = set(candidate_object.columns)

        if REQUIRED_TUNING_COLUMNS.issubset(candidate_columns):
            source_training_df = candidate_object
            source_training_df_name = candidate_name
            break

if source_training_df is None:
    raise ValueError(
        "A prepared training DataFrame could not be located. "
        "The DataFrame must contain FL_DATE, features, and "
        f"{TARGET_COLUMN}. Add its variable name to "
        "SOURCE_DATAFRAME_CANDIDATES."
    )

print(f"Tuning source DataFrame: {source_training_df_name}")


# ============================================================
# 3. Prepare the stratified tuning dataset
# ============================================================

# The majority on-time class is sampled at 10%, while the
# delayed-flight class is sampled at 30%. FL_DATE is retained
# for chronological fold filtering.

TUNING_TABLE = "workspace.default.flight_delay_tree_tuning"

prepared_tuning_df = (
    source_training_df
    .select(
        F.to_date(F.col("FL_DATE")).alias("FL_DATE"),
        F.col(TARGET_COLUMN).cast("double").alias(TARGET_COLUMN),
        F.col("features"),
    )
    .filter(
        F.col("FL_DATE").isNotNull()
        & F.col(TARGET_COLUMN).isin(0.0, 1.0)
        & F.col("features").isNotNull()
    )
    .sampleBy(
        col=TARGET_COLUMN,
        fractions={
            0.0: 0.10,
            1.0: 0.30,
        },
        seed=RANDOM_SEED,
    )
)

(
    prepared_tuning_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TUNING_TABLE)
)

df_train_tree_tuning = spark.table(TUNING_TABLE)

tuning_row_count = df_train_tree_tuning.count()

if tuning_row_count == 0:
    raise ValueError(
        "The stratified tuning dataset contains zero rows. "
        "Verify the source DataFrame and target-column values."
    )

print(f"Tuning table created: {TUNING_TABLE}")
print(f"Tree-model tuning rows: {tuning_row_count:,}")


# ============================================================
# 4. Chronological expanding-window fold definitions
# ============================================================

TUNING_FOLDS = [
    {
        "fold": "Fold 1",
        "train_end": "2025-04-30",
        "validation_start": "2025-05-01",
        "validation_end": "2025-05-31",
    },
    {
        "fold": "Fold 2",
        "train_end": "2025-05-31",
        "validation_start": "2025-06-01",
        "validation_end": "2025-06-30",
    },
    {
        "fold": "Fold 3",
        "train_end": "2025-06-30",
        "validation_start": "2025-07-01",
        "validation_end": "2025-07-31",
    },
    {
        "fold": "Fold 4",
        "train_end": "2025-07-31",
        "validation_start": "2025-08-01",
        "validation_end": "2025-08-31",
    },
]

print(f"Chronological tuning folds: {len(TUNING_FOLDS)}")


# ============================================================
# 5. Prediction evaluation function
# ============================================================

def evaluate_tuning_predictions(
    predictions: DataFrame,
) -> dict[str, float]:
    """Calculate overall and delayed-flight validation metrics."""

    accuracy = MulticlassClassificationEvaluator(
        labelCol=TARGET_COLUMN,
        predictionCol="prediction",
        metricName="accuracy",
    ).evaluate(predictions)

    precision = MulticlassClassificationEvaluator(
        labelCol=TARGET_COLUMN,
        predictionCol="prediction",
        metricName="weightedPrecision",
    ).evaluate(predictions)

    recall = MulticlassClassificationEvaluator(
        labelCol=TARGET_COLUMN,
        predictionCol="prediction",
        metricName="weightedRecall",
    ).evaluate(predictions)

    f1_score = MulticlassClassificationEvaluator(
        labelCol=TARGET_COLUMN,
        predictionCol="prediction",
        metricName="f1",
    ).evaluate(predictions)

    delay_precision = MulticlassClassificationEvaluator(
        labelCol=TARGET_COLUMN,
        predictionCol="prediction",
        metricName="precisionByLabel",
        metricLabel=1.0,
    ).evaluate(predictions)

    delay_recall = MulticlassClassificationEvaluator(
        labelCol=TARGET_COLUMN,
        predictionCol="prediction",
        metricName="recallByLabel",
        metricLabel=1.0,
    ).evaluate(predictions)

    delay_f1 = (
        2.0 * delay_precision * delay_recall
        / (delay_precision + delay_recall)
        if delay_precision + delay_recall > 0
        else 0.0
    )

    roc_auc = BinaryClassificationEvaluator(
        labelCol=TARGET_COLUMN,
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC",
    ).evaluate(predictions)

    pr_auc = BinaryClassificationEvaluator(
        labelCol=TARGET_COLUMN,
        rawPredictionCol="rawPrediction",
        metricName="areaUnderPR",
    ).evaluate(predictions)

    return {
        "ACCURACY": float(accuracy),
        "PRECISION": float(precision),
        "RECALL": float(recall),
        "F1_SCORE": float(f1_score),
        "ROC_AUC": float(roc_auc),
        "PR_AUC": float(pr_auc),
        "DELAY_PRECISION": float(delay_precision),
        "DELAY_RECALL": float(delay_recall),
        "DELAY_F1": float(delay_f1),
    }


# ============================================================
# 6. Fold-level training and evaluation function
# ============================================================

def train_and_evaluate_fold(
    estimator,
    training_df: DataFrame,
    validation_df: DataFrame,
) -> dict[str, float]:
    """Train one estimator and evaluate one chronological fold."""

    if training_df.limit(1).count() == 0:
        raise ValueError("A chronological training fold contains zero rows.")

    if validation_df.limit(1).count() == 0:
        raise ValueError("A chronological validation fold contains zero rows.")

    training_start_time = time.perf_counter()

    fitted_model = estimator.fit(training_df)

    training_seconds = time.perf_counter() - training_start_time

    predictions = fitted_model.transform(validation_df)

    metrics = evaluate_tuning_predictions(predictions)
    metrics["TRAINING_SECONDS"] = float(training_seconds)

    return metrics


# ============================================================
# 7. Reusable hyperparameter-tuning function
# ============================================================

def tune_model(
    model_name: str,
    estimator_builder: Callable,
    parameter_grid: list[dict],
    tuning_df: DataFrame = df_train_tree_tuning,
    tuning_folds: list[dict] = TUNING_FOLDS,
) -> DataFrame:
    """Tune one model across chronological validation folds."""

    if not parameter_grid:
        raise ValueError("The parameter grid cannot be empty.")

    tuning_results = []

    for configuration_number, params in enumerate(
        parameter_grid,
        start=1,
    ):
        fold_metrics = []

        print(
            f"Evaluating {model_name} configuration "
            f"{configuration_number}/{len(parameter_grid)}: {params}"
        )

        for fold in tuning_folds:
            training_df = tuning_df.filter(
                F.col("FL_DATE")
                <= F.to_date(F.lit(fold["train_end"]))
            )

            validation_df = tuning_df.filter(
                F.col("FL_DATE").between(
                    F.to_date(
                        F.lit(fold["validation_start"])
                    ),
                    F.to_date(
                        F.lit(fold["validation_end"])
                    ),
                )
            )

            estimator = estimator_builder(**params)

            metrics = train_and_evaluate_fold(
                estimator=estimator,
                training_df=training_df,
                validation_df=validation_df,
            )

            fold_metrics.append(metrics)

        fold_count = len(fold_metrics)

        if fold_count == 0:
            raise ValueError(
                f"No fold metrics were produced for {model_name}."
            )

        tuning_results.append(
            {
                "MODEL": model_name,
                "PARAMETERS": str(params),

                "ACCURACY": round(
                    sum(
                        item["ACCURACY"]
                        for item in fold_metrics
                    ) / fold_count,
                    4,
                ),
                "PRECISION": round(
                    sum(
                        item["PRECISION"]
                        for item in fold_metrics
                    ) / fold_count,
                    4,
                ),
                "RECALL": round(
                    sum(
                        item["RECALL"]
                        for item in fold_metrics
                    ) / fold_count,
                    4,
                ),
                "F1_SCORE": round(
                    sum(
                        item["F1_SCORE"]
                        for item in fold_metrics
                    ) / fold_count,
                    4,
                ),
                "ROC_AUC": round(
                    sum(
                        item["ROC_AUC"]
                        for item in fold_metrics
                    ) / fold_count,
                    4,
                ),
                "PR_AUC": round(
                    sum(
                        item["PR_AUC"]
                        for item in fold_metrics
                    ) / fold_count,
                    4,
                ),
                "DELAY_PRECISION": round(
                    sum(
                        item["DELAY_PRECISION"]
                        for item in fold_metrics
                    ) / fold_count,
                    4,
                ),
                "DELAY_RECALL": round(
                    sum(
                        item["DELAY_RECALL"]
                        for item in fold_metrics
                    ) / fold_count,
                    4,
                ),
                "DELAY_F1": round(
                    sum(
                        item["DELAY_F1"]
                        for item in fold_metrics
                    ) / fold_count,
                    4,
                ),
                "TRAINING_SECONDS": round(
                    sum(
                        item["TRAINING_SECONDS"]
                        for item in fold_metrics
                    ) / fold_count,
                    2,
                ),
            }
        )

    return (
        spark.createDataFrame(tuning_results)
        .select(
            "MODEL",
            "PARAMETERS",
            "ACCURACY",
            "PRECISION",
            "RECALL",
            "F1_SCORE",
            "ROC_AUC",
            "PR_AUC",
            "DELAY_PRECISION",
            "DELAY_RECALL",
            "DELAY_F1",
            "TRAINING_SECONDS",
        )
        .orderBy(
            F.desc("DELAY_RECALL"),
            F.desc("PR_AUC"),
            F.desc("DELAY_F1"),
            F.desc("ROC_AUC"),
        )
    )


print("Self-contained chronological tuning framework created successfully.")

### 6.5 Logistic Regression Hyperparameter Tuning

Logistic Regression is optimized by evaluating multiple combinations of regularization strength and elastic-net mixing.

These hyperparameters control model complexity and reduce the risk of overfitting while maintaining good generalization performance.

Each candidate configuration is evaluated using the chronological validation folds defined previously. The average Accuracy, weighted Precision, weighted Recall, weighted F1-score, delayed-flight Precision, delayed-flight Recall, delayed-flight F1-score, ROC AUC, and Precision–Recall AUC are calculated for every configuration.

The configurations are compared using the complete metric set, with particular attention given to delayed-flight detection performance.

In [0]:
from pyspark.ml.classification import LogisticRegression


LOGISTIC_REGRESSION_PARAMETER_GRID = [
    {
        "regParam": 0.001,
        "elasticNetParam": 0.0,
    },
    {
        "regParam": 0.01,
        "elasticNetParam": 0.0,
    },
    {
        "regParam": 0.10,
        "elasticNetParam": 0.0,
    },
    {
        "regParam": 0.01,
        "elasticNetParam": 0.5,
    },
    {
        "regParam": 0.01,
        "elasticNetParam": 1.0,
    },
]

print(
    f"Candidate Logistic Regression configurations: "
    f"{len(LOGISTIC_REGRESSION_PARAMETER_GRID)}"
)

#### Logistic Regression Estimator Builder

A reusable estimator builder is created for Logistic Regression.

Rather than manually constructing a new Logistic Regression model for every hyperparameter configuration, this function generates a configured estimator using the supplied tuning parameters.

Only the hyperparameters under evaluation (`regParam` and `elasticNetParam`) vary between candidate configurations. All remaining model settings remain fixed to ensure that performance differences are attributable solely to the tuned parameters.

This reusable design allows the same chronological tuning workflow to evaluate every Logistic Regression configuration consistently while minimizing duplicated code.

In [0]:
def build_logistic_regression(
    regParam,
    elasticNetParam,
):

    return LogisticRegression(
        featuresCol="features",
        labelCol=TARGET_COLUMN,
        predictionCol="prediction",
        probabilityCol="probability",
        rawPredictionCol="rawPrediction",

        regParam=regParam,
        elasticNetParam=elasticNetParam,

        maxIter=20,
        standardization=True,
        family="binomial",
    )


print("Logistic Regression estimator builder created successfully.")

#### Tune Logistic Regression

The predefined Logistic Regression hyperparameter configurations are evaluated using the reusable chronological tuning workflow.

Each candidate configuration is trained using the chronological training folds and evaluated on the corresponding validation folds.

The average delayed-flight Precision, Recall, F1-score, ROC Area Under the Curve (ROC AUC), and Precision–Recall Area Under the Curve (PR AUC) are calculated to identify the best-performing Logistic Regression configuration.

In [0]:
lr_tuning_results = tune_model(
    model_name="Logistic Regression",
    estimator_builder=build_logistic_regression,
    parameter_grid=LOGISTIC_REGRESSION_PARAMETER_GRID,
)

display(lr_tuning_results)

### Logistic Regression Tuning Results

Five Logistic Regression configurations were evaluated using four chronological expanding-window validation folds. The reported metrics represent the average performance across those folds.

The best-performing configuration used:

- `regParam = 0.001`
- `elasticNetParam = 0.0`

This configuration achieved:

- Accuracy of **0.6284**
- Weighted Precision of **0.6442**
- Weighted Recall of **0.6284**
- Weighted F1-score of **0.6155**
- ROC AUC of **0.6850**
- PR AUC of **0.6643**
- Delayed-flight Precision of **0.6108**
- Delayed-flight Recall of **0.7536**
- Delayed-flight F1-score of **0.6674**

Among the evaluated configurations, this model produced the highest delayed-flight Recall and delayed-flight F1-score. It also achieved the strongest ROC AUC and PR AUC.

The results indicate that weak L2 regularization provides the best balance between overall classification performance and delayed-flight detection. Stronger regularization and elastic-net penalties reduced delayed-flight Recall and delayed-flight F1-score.

This configuration is retained as the tuned Logistic Regression candidate for comparison with the tuned Random Forest and Gradient-Boosted Trees models.

### 6.6 Random Forest Hyperparameter Tuning

Random Forest is optimized by evaluating multiple combinations of the number of trees and maximum tree depth.

The number of trees controls the size of the ensemble, while maximum depth controls the complexity of each individual decision tree. Increasing these values may improve predictive performance, but it also increases training time and the risk of overfitting.

Each candidate configuration is evaluated using the same four chronological expanding-window validation folds used for Logistic Regression.

The reported metrics represent the average performance across those folds. The configurations are compared using Accuracy, weighted Precision, weighted Recall, weighted F1-score, delayed-flight Precision, delayed-flight Recall, delayed-flight F1-score, ROC AUC, PR AUC, and average training time.

In [0]:
from pyspark.ml.classification import RandomForestClassifier


RANDOM_FOREST_PARAMETER_GRID = [
    {
        "numTrees": 5,
        "maxDepth": 4,
    },
    {
        "numTrees": 5,
        "maxDepth": 6,
    },
    {
        "numTrees": 10,
        "maxDepth": 4,
    },
    {
        "numTrees": 10,
        "maxDepth": 6,
    },
]


print(
    f"Candidate Random Forest configurations: "
    f"{len(RANDOM_FOREST_PARAMETER_GRID)}"
)

### Random Forest Estimator Builder

A reusable estimator builder is created for Random Forest classification.

The function constructs a new Random Forest estimator for each combination of `numTrees` and `maxDepth` in the parameter grid.

Only the hyperparameters being evaluated vary between configurations. The remaining settings are held constant so that performance differences can be attributed to the selected number of trees and tree depth.

The estimator is designed to work with the same chronological tuning workflow used for Logistic Regression.

In [0]:
def build_random_forest(
    numTrees,
    maxDepth,
):
    """Build a configured Random Forest estimator."""

    return RandomForestClassifier(
        featuresCol="features",
        labelCol=TARGET_COLUMN,
        predictionCol="prediction",
        probabilityCol="probability",
        rawPredictionCol="rawPrediction",
        numTrees=numTrees,
        maxDepth=maxDepth,
        seed=42,
    )


print("Random Forest estimator builder created successfully.")

### Tune Random Forest

The predefined Random Forest hyperparameter configurations are evaluated using the reusable chronological tuning workflow.

Each candidate configuration is trained across the four chronological expanding-window folds and evaluated on the corresponding validation folds.

The reported metrics represent the average performance across the folds. The configurations are compared using overall classification performance, delayed-flight detection performance, ROC AUC, PR AUC, and average training time.

In [0]:
rf_tuning_results = tune_model(
    model_name="Random Forest",
    estimator_builder=build_random_forest,
    parameter_grid=RANDOM_FOREST_PARAMETER_GRID,
)

display(rf_tuning_results)

### Random Forest Tuning Results

Four Random Forest configurations were evaluated using four chronological expanding-window validation folds. The reported metrics represent the average performance across those folds.

The best-performing configuration used:

- `numTrees = 10`
- `maxDepth = 6`

This configuration achieved:

- Accuracy of **0.5643**
- Weighted Precision of **0.6229**
- Weighted Recall of **0.5643**
- Weighted F1-score of **0.4873**
- ROC AUC of **0.6718**
- PR AUC of **0.6507**
- Delayed-flight Precision of **0.6845**
- Delayed-flight Recall of **0.3037**
- Delayed-flight F1-score of **0.3277**

Increasing both the number of trees and the maximum tree depth improved Random Forest performance compared with the other evaluated configurations. However, despite these improvements, the tuned Random Forest model produced substantially lower delayed-flight Recall and delayed-flight F1-score than the tuned Logistic Regression model.

The tuned Random Forest configuration is retained for comparison with the tuned Logistic Regression and Gradient-Boosted Trees models during final model selection.

### 6.7 Gradient-Boosted Trees Hyperparameter Tuning

Gradient-Boosted Trees are optimized by evaluating multiple combinations of boosting iterations, maximum tree depth, and learning rate.

The number of iterations controls how many trees are added sequentially, maximum depth controls the complexity of each tree, and the learning rate determines how strongly each new tree contributes to the final model.

Each candidate configuration is evaluated using the same four chronological expanding-window validation folds used for Logistic Regression and Random Forest.

The reported metrics represent the average performance across the folds. The configurations are compared using overall classification performance, delayed-flight detection performance, ROC AUC, PR AUC, and average training time.

In [0]:
from pyspark.ml.classification import GBTClassifier


GBT_PARAMETER_GRID = [
    {
        "maxIter": 3,
        "maxDepth": 3,
        "stepSize": 0.10,
    },
    {
        "maxIter": 5,
        "maxDepth": 3,
        "stepSize": 0.10,
    },
    {
        "maxIter": 5,
        "maxDepth": 4,
        "stepSize": 0.10,
    },
]


print(
    f"Candidate Gradient-Boosted Trees configurations: "
    f"{len(GBT_PARAMETER_GRID)}"
)

#### Gradient-Boosted Trees Estimator Builder

A reusable estimator builder is created for Gradient-Boosted Trees classification.

The function constructs a new estimator for each combination of `maxIter`, `maxDepth`, and `stepSize` in the parameter grid.

Only the tuned hyperparameters vary between candidate configurations. The remaining settings are held constant so that performance differences can be attributed to the number of boosting iterations, tree depth, and learning rate.

The estimator is compatible with the same chronological tuning workflow used for Logistic Regression and Random Forest.

In [0]:
def build_gradient_boosted_trees(
    maxIter,
    maxDepth,
    stepSize,
):
    """Build a configured Gradient-Boosted Trees estimator."""

    return GBTClassifier(
        featuresCol="features",
        labelCol=TARGET_COLUMN,
        predictionCol="prediction",
        probabilityCol="probability",
        rawPredictionCol="rawPrediction",
        maxIter=maxIter,
        maxDepth=maxDepth,
        stepSize=stepSize,
        seed=42,
    )


print("Gradient-Boosted Trees estimator builder created successfully.")

#### Tune Gradient-Boosted Trees

The predefined Gradient-Boosted Trees hyperparameter configurations are evaluated using the reusable chronological tuning workflow.

Each candidate configuration is trained across the four chronological expanding-window folds and evaluated on the corresponding validation folds.

The reported metrics represent the average performance across the folds. The configurations are compared using overall classification performance, delayed-flight detection performance, ROC AUC, PR AUC, and average training time.

In [0]:
def build_gradient_boosted_trees(
    maxIter,
    maxDepth,
    stepSize,
):
    """Build a configured Gradient-Boosted Trees estimator."""

    return GBTClassifier(
        featuresCol="features",
        labelCol=TARGET_COLUMN,
        predictionCol="prediction",
        maxIter=maxIter,
        maxDepth=maxDepth,
        stepSize=stepSize,
        seed=42,
    )


print("Gradient-Boosted Trees estimator builder created successfully.")

In [0]:
gbt_tuning_results = tune_model(
    model_name="Gradient-Boosted Trees",
    estimator_builder=build_gradient_boosted_trees,
    parameter_grid=GBT_PARAMETER_GRID,
)

display(gbt_tuning_results)

### Gradient-Boosted Trees Tuning Results

Three Gradient-Boosted Trees configurations were evaluated using four chronological expanding-window validation folds. The reported metrics represent the average performance across those folds.

The best-performing configuration used:

- `maxIter = 5`
- `maxDepth = 4`
- `stepSize = 0.10`

This configuration achieved:

- Accuracy of **0.6311**
- Weighted Precision of **0.6377**
- Weighted Recall of **0.6311**
- Weighted F1-score of **0.6275**
- ROC AUC of **0.6815**
- PR AUC of **0.6622**
- Delayed-flight Precision of **0.6425**
- Delayed-flight Recall of **0.6294**
- Delayed-flight F1-score of **0.6287**

Among the evaluated configurations, this model achieved the strongest overall classification performance while maintaining competitive delayed-flight detection performance.

Although the tuned Gradient-Boosted Trees model slightly outperformed the other models in overall Accuracy and F1-score, its delayed-flight Recall remained lower than that of the tuned Logistic Regression model. Since the primary objective of this project is to identify as many delayed flights as possible for operational prioritization, the tuned Gradient-Boosted Trees model is retained for comparison during the final model selection stage.

### 6.8 Tuned Model Comparison

The best-performing configuration from each machine learning algorithm is selected and compared using the average metrics obtained from the chronological hyperparameter tuning process.

The comparison considers both overall classification performance and delayed-flight detection performance. Since the objective of this project is to prioritize flights that are likely to experience arrival delays of at least 15 minutes, particular emphasis is placed on delayed-flight Recall and delayed-flight F1-score while also considering ROC AUC, PR AUC, and overall model performance.

This comparison provides the basis for selecting the final predictive model for operational deployment.

In [0]:
from pyspark.sql import functions as F

best_logistic_regression = (
    lr_tuning_results
    .limit(1)
)

best_random_forest = (
    rf_tuning_results
    .limit(1)
)

best_gradient_boosted_trees = (
    gbt_tuning_results
    .limit(1)
)

tuned_model_comparison = (
    best_logistic_regression
    .unionByName(best_random_forest)
    .unionByName(best_gradient_boosted_trees)
)

display(tuned_model_comparison)

### Tuned Model Comparison Results

The best-performing configuration from each machine learning algorithm was compared using the average performance across the chronological validation folds.

The comparison shows that:

- **Logistic Regression** achieved the highest delayed-flight Recall (0.7536), delayed-flight F1-score (0.6674), ROC AUC (0.6850), and PR AUC (0.6643). These results indicate that Logistic Regression provides the strongest ability to identify flights that are likely to experience significant arrival delays.

- **Random Forest** achieved the highest delayed-flight Precision (0.6845), indicating that when the model predicts a delay, it is more likely to be correct. However, its substantially lower delayed-flight Recall (0.3037) indicates that many delayed flights would not be identified, limiting its usefulness for operational prioritization.

- **Gradient-Boosted Trees** achieved the highest overall Accuracy (0.6311) and weighted F1-score (0.6275), demonstrating balanced overall classification performance. However, its delayed-flight Recall (0.6294) remained lower than that of Logistic Regression.

Overall, the comparison demonstrates that although Gradient-Boosted Trees provide the strongest overall classification performance, Logistic Regression delivers the best delayed-flight detection performance, which more closely aligns with the objective of proactively identifying flights requiring operational attention.

## 7. Evaluate Model Performance

### Purpose

The tuned candidate models are evaluated to determine which algorithm provides the most effective balance between overall predictive performance and delayed-flight detection.

Although several evaluation metrics are considered, the primary objective of this project is to identify flights that are likely to arrive at least 15 minutes late before departure. Consequently, delayed-flight Recall and delayed-flight F1-score receive greater emphasis than overall Accuracy alone, since failing to identify delayed flights would reduce the operational value of the prediction system.

The evaluation considers:

- Overall Accuracy
- Weighted Precision
- Weighted Recall
- Weighted F1-score
- ROC Area Under the Curve (ROC AUC)
- Precision–Recall Area Under the Curve (PR AUC)
- Delayed-flight Precision
- Delayed-flight Recall
- Delayed-flight F1-score

In [0]:
display(tuned_model_comparison)

### Final Model Evaluation

The comparison demonstrates that each machine learning algorithm exhibits different strengths.

Gradient-Boosted Trees achieved the highest overall Accuracy and weighted F1-score, indicating the strongest overall classification performance across all observations.

Random Forest achieved the highest delayed-flight Precision, indicating that flights predicted as delayed were more likely to truly experience delays. However, its substantially lower delayed-flight Recall indicates that many delayed flights were not identified.

Logistic Regression achieved the highest delayed-flight Recall, delayed-flight F1-score, ROC AUC, and PR AUC. These metrics indicate that Logistic Regression provides the strongest ability to identify flights at risk of significant arrival delays while maintaining competitive overall predictive performance.

Since the objective of this project is to prioritize potentially delayed flights before departure, maximizing delayed-flight detection is more important than maximizing overall Accuracy alone. Consequently, Logistic Regression demonstrates the strongest operational performance among the evaluated models.

### Model Strengths and Weaknesses

### Logistic Regression

**Strengths**

- Highest delayed-flight Recall.
- Highest delayed-flight F1-score.
- Highest ROC AUC.
- Highest PR AUC.
- Fastest training time.

**Limitations**

- Slightly lower overall Accuracy than Gradient-Boosted Trees.

---

### Random Forest

**Strengths**

- Highest delayed-flight Precision.
- Robust ensemble learning approach.

**Limitations**

- Lowest delayed-flight Recall.
- Misses a large proportion of delayed flights.

---

### Gradient-Boosted Trees

**Strengths**

- Highest overall Accuracy.
- Highest weighted F1-score.
- Strong overall classification performance.

**Limitations**

- Significantly longer training time.
- Lower delayed-flight Recall than Logistic Regression.

## 8. Select the Best Model

### Final Model Selection Criteria

The final predictive model is selected based on both statistical performance and alignment with the operational objective of the project.

Although overall Accuracy, weighted Precision, weighted Recall, and weighted F1-score are considered, the primary objective is to identify flights that are likely to arrive at least 15 minutes late so they can be prioritized for operational review.

For this reason, delayed-flight Recall and delayed-flight F1-score receive greater emphasis during final model selection. ROC AUC, PR AUC, training efficiency, and overall classification performance are also considered to ensure that the selected model provides reliable and practical predictive performance.

In [0]:
SELECTED_MODEL_NAME = "Logistic Regression"

SELECTED_MODEL_PARAMETERS = {
    "regParam": 0.001,
    "elasticNetParam": 0.0,
}

selected_model_summary = (
    tuned_model_comparison
    .filter(F.col("MODEL") == SELECTED_MODEL_NAME)
)

display(selected_model_summary)

print(f"Selected model: {SELECTED_MODEL_NAME}")
print(f"Selected parameters: {SELECTED_MODEL_PARAMETERS}")

### Final Model Selection Decision

Logistic Regression is selected as the final predictive model.

The tuned Logistic Regression configuration used:

- `regParam = 0.001`
- `elasticNetParam = 0.0`

This model achieved the highest delayed-flight Recall of **0.7536**, meaning it identified approximately 75% of delayed flights across the chronological validation folds. It also achieved the highest delayed-flight F1-score of **0.6674**, ROC AUC of **0.6850**, and PR AUC of **0.6643** among the tuned candidate models.

Gradient-Boosted Trees achieved slightly higher overall Accuracy and weighted F1-score, while Random Forest achieved higher delayed-flight Precision. However, both models produced lower delayed-flight Recall than Logistic Regression.

Because the business objective is to proactively identify flights requiring operational attention, missing a delayed flight represents a more significant operational risk than reviewing an additional flight that ultimately arrives on time.

Logistic Regression therefore provides the strongest alignment between predictive performance, delayed-flight detection, computational efficiency, and operational value.

## 9. Save the Selected Model and Supporting Artifacts

The selected Logistic Regression configuration is retrained before being saved as a Spark ML model.

The final training dataset combines the original training and validation periods. This allows the selected model to learn from all historical observations available before the test period while preserving the test dataset for final holdout evaluation.

The saved artifacts will include:

- The fitted Spark ML Logistic Regression model
- The selected hyperparameters
- Model evaluation metrics
- Dataset and feature metadata

Saving these artifacts supports reproducibility and allows the model to be loaded later for batch inference, dashboard integration, and explainability analysis.

### 9.1 Train the Final Selected Model Using the Tuning Sampling Strategy

The selected Logistic Regression configuration is trained using the combined training and validation periods while preserving the same stratified sampling strategy used during chronological hyperparameter tuning.

The on-time class is sampled at 10%, while the delayed-flight class is sampled at 30%. This maintains consistency between model selection and final training and reduces the tendency of the model to predict nearly all observations as on-time.

The untouched test dataset remains unsampled and is used only for final holdout evaluation.

In [0]:
from pyspark.ml.classification import LogisticRegression
from pyspark.sql import functions as F


# Combine the chronological training and validation periods.
df_final_training_full = (
    df_train_hashed
    .unionByName(df_validation_hashed)
)


# Apply the same class-specific sampling strategy used during tuning.
df_final_training_sampled = (
    df_final_training_full
    .sampleBy(
        col=TARGET_COLUMN,
        fractions={
            0.0: 0.10,
            1.0: 0.30,
        },
        seed=42,
    )
)


# Validate the sampled final-training distribution.
print("Sampled final-training class distribution:")

df_final_training_sampled.groupBy(
    TARGET_COLUMN
).count().orderBy(
    TARGET_COLUMN
).show()


# Create the selected Logistic Regression estimator.
final_logistic_regression_estimator = LogisticRegression(
    featuresCol="features",
    labelCol=TARGET_COLUMN,
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    regParam=0.001,
    elasticNetParam=0.0,
    maxIter=20,
    standardization=True,
    family="binomial",
)


# Train the final selected model.
final_selected_model = final_logistic_regression_estimator.fit(
    df_final_training_sampled
)


print("Final sampled Logistic Regression model trained successfully.")
print("Selected regParam: 0.001")
print("Selected elasticNetParam: 0.0")

### Final Holdout Test Evaluation

The retrained Logistic Regression model is evaluated using the untouched November–December 2025 holdout test dataset.

Unlike the chronological validation folds used during hyperparameter tuning, the holdout dataset was never used during model development, parameter selection, or final model training. Consequently, this evaluation provides the most realistic estimate of how the selected model is expected to perform on future unseen flight records.

The same evaluation metrics used throughout the project are calculated to support direct comparison between the tuning results and the final deployment-ready model.

In [0]:
final_test_predictions = final_selected_model.transform(
    df_test_hashed
)

FINAL_TEST_METRICS = evaluate_tuning_predictions(
    final_test_predictions
)

display(
    spark.createDataFrame(
        [
            {
                "MODEL": "Logistic Regression",
                "DATASET": "Holdout Test",
                **{
                    metric_name: round(metric_value, 4)
                    for metric_name, metric_value
                    in FINAL_TEST_METRICS.items()
                },
            }
        ]
    )
)

### Holdout Test Evaluation

The final Logistic Regression model was evaluated on the untouched November–December 2025 test dataset.

Compared with the chronological validation results obtained during hyperparameter tuning, the holdout evaluation showed lower delayed-flight Recall and delayed-flight F1-score.

This reduction is expected because the holdout dataset contains completely unseen observations and therefore provides a more realistic estimate of operational performance.

Despite the reduction, the selected Logistic Regression model maintained the strongest balance between overall predictive performance and delayed-flight detection among the evaluated candidate models.

### Holdout Test Results

The final Logistic Regression model achieved the following performance on the unseen holdout dataset:

- Accuracy: **0.6689**
- Weighted Precision: **0.6948**
- Weighted Recall: **0.6689**
- Weighted F1-score: **0.6800**
- ROC AUC: **0.6320**
- PR AUC: **0.3310**
- Delayed-flight Precision: **0.3395**
- Delayed-flight Recall: **0.4190**
- Delayed-flight F1-score: **0.3751**

Compared with the chronological validation results obtained during hyperparameter tuning, the holdout evaluation shows a reduction in delayed-flight detection performance. This reduction is expected because the holdout dataset contains completely unseen observations and therefore provides a more realistic estimate of operational performance.

Despite the decrease in delayed-flight Recall, the Logistic Regression model maintained balanced overall predictive performance and continued to outperform the alternative models during model selection. The holdout evaluation therefore provides confidence that the selected model can generalize to new flight records while acknowledging the inherent challenges of predicting rare delay events.

### 9.2 Save the Trained Spark ML Model

The retrained Logistic Regression model is saved as a native Spark ML model.

Saving the model enables future loading for batch inference, operational deployment, dashboard integration, and explainability analysis without requiring retraining.

The model is stored in the Databricks workspace using Spark ML's native persistence format.

In [0]:
from pathlib import Path

MODEL_SAVE_PATH = (
    "/Volumes/workspace/default/flight_delay_capstone/models/"
    "logistic_regression_final"
)

final_selected_model.write().overwrite().save(
    MODEL_SAVE_PATH
)

print("Final model saved successfully.")
print(f"Location: {MODEL_SAVE_PATH}")

### 9.3 Save Model Metadata

The selected hyperparameters and evaluation metrics are saved to support reproducibility and future model maintenance.

These metadata describe the final production model without requiring the tuning process to be rerun.

In [0]:
import json

selected_metrics_row = (
    selected_model_summary
    .select(
        "MODEL",
        "PARAMETERS",
        "ACCURACY",
        "PRECISION",
        "RECALL",
        "F1_SCORE",
        "ROC_AUC",
        "PR_AUC",
        "DELAY_PRECISION",
        "DELAY_RECALL",
        "DELAY_F1",
        "TRAINING_SECONDS",
    )
    .first()
)

if selected_metrics_row is None:
    raise ValueError(
        "Selected model metadata could not be retrieved."
    )

MODEL_METADATA = {
    "model": selected_metrics_row["MODEL"],
    "target": TARGET_COLUMN,
    "parameters": SELECTED_MODEL_PARAMETERS,
    "parameter_summary": selected_metrics_row["PARAMETERS"],
    "hash_vector_size": HASH_VECTOR_SIZE,
    "categorical_features": CATEGORICAL_COLUMNS,
    "numerical_features": NUMERICAL_COLUMNS,
    "total_raw_predictors": len(MODEL_INPUT_COLUMNS),
    "selected_after_hyperparameter_tuning": True,
    "final_training_period": "January 2025 to October 2025",
    "holdout_test_period": "November 2025 to December 2025",
    "model_format": "Spark ML",
}

dbutils.fs.put(
    (
        "/Volumes/workspace/default/flight_delay_capstone/models/"
        "logistic_regression_final/model_metadata.json"
    ),
    json.dumps(MODEL_METADATA, indent=4),
    overwrite=True,
)

print("Dynamic model metadata saved successfully.")

### 9.4 Save Evaluation Metrics

The final evaluation metrics of the selected model are saved to provide a permanent record of the model's predictive performance.

These metrics support future model monitoring and performance comparison.

In [0]:
MODEL_METRICS = {
    "accuracy": float(selected_metrics_row["ACCURACY"]),
    "precision": float(selected_metrics_row["PRECISION"]),
    "recall": float(selected_metrics_row["RECALL"]),
    "f1_score": float(selected_metrics_row["F1_SCORE"]),
    "roc_auc": float(selected_metrics_row["ROC_AUC"]),
    "pr_auc": float(selected_metrics_row["PR_AUC"]),
    "delay_precision": float(
        selected_metrics_row["DELAY_PRECISION"]
    ),
    "delay_recall": float(
        selected_metrics_row["DELAY_RECALL"]
    ),
    "delay_f1": float(
        selected_metrics_row["DELAY_F1"]
    ),
    "average_training_seconds": float(
        selected_metrics_row["TRAINING_SECONDS"]
    ),
    "metric_source": (
        "Average performance across four chronological "
        "hyperparameter-tuning validation folds"
    ),
}

dbutils.fs.put(
    (
        "/Volumes/workspace/default/flight_delay_capstone/models/"
        "logistic_regression_final/model_metrics.json"
    ),
    json.dumps(MODEL_METRICS, indent=4),
    overwrite=True,
)

print("Dynamic model evaluation metrics saved successfully.")

## 10. Explainable Artificial Intelligence (SHAP)

### Purpose

Although the production prediction model uses Spark's `FeatureHasher` for scalable preprocessing, hashed feature vectors do not preserve a direct mapping between vector positions and the original predictor names. Consequently, the hashed feature space is not suitable for producing human-interpretable SHAP explanations.

To provide meaningful global and local explanations, a separate explainability workflow is created using the original human-readable predictors. A surrogate Logistic Regression model is trained on the explainability dataset using the same target variable as the production model.

The surrogate model is used exclusively for SHAP analysis and does not replace the production prediction model.

This approach enables interpretable feature importance rankings, SHAP summary plots, dependence plots, and individual flight explanations while maintaining consistency with the production prediction pipeline.

### 10.1 Prepare the Explainability Dataset

The explainability dataset is created from the original holdout dataset using the human-readable predictor variables.

Unlike the production pipeline, no feature hashing is applied because SHAP requires identifiable feature names in order to produce meaningful explanations.

In [0]:
EXPLAINABILITY_COLUMNS = (
    MODEL_INPUT_COLUMNS
    + [TARGET_COLUMN]
)

explainability_df = (
    df_test_hist
    .select(*EXPLAINABILITY_COLUMNS)
    .dropna()
)

print(
    f"Explainability rows: "
    f"{explainability_df.count():,}"
)

In [0]:
import pandas as pd

SHAP_SAMPLE_SIZE = 20000
SHAP_RANDOM_SEED = 42

explainability_sample = (
    explainability_df
    .orderBy(F.rand(seed=SHAP_RANDOM_SEED))
    .limit(SHAP_SAMPLE_SIZE)
)

explainability_pd = explainability_sample.toPandas()

print(f"Explainability sample rows: {len(explainability_pd):,}")
print(f"Explainability columns: {len(explainability_pd.columns)}")

display(explainability_sample.limit(5))

### 10.2 Train the Explainability Logistic Regression Model

The sampled explainability dataset is divided into predictor and target variables.

Categorical variables are transformed using one-hot encoding, while numerical variables are passed through without alteration. The resulting feature matrix preserves identifiable feature names so that SHAP values can be mapped back to human-readable predictors.

A scikit-learn Logistic Regression model is then trained solely for explainability. This model supports SHAP analysis and does not replace the production Spark ML prediction model.

In [0]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# Separate predictors and target.
X_explainability = explainability_pd[
    MODEL_INPUT_COLUMNS
].copy()

y_explainability = explainability_pd[
    TARGET_COLUMN
].astype(int).copy()


# Scale numerical predictors while preserving sparse compatibility.
numerical_transformer = Pipeline(
    steps=[
        (
            "scaler",
            StandardScaler(with_mean=False),
        ),
    ]
)


# Configure readable preprocessing.
explainability_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True,
            ),
            CATEGORICAL_COLUMNS,
        ),
        (
            "numerical",
            numerical_transformer,
            NUMERICAL_COLUMNS,
        ),
    ],
    remainder="drop",
)


# Configure the explainability Logistic Regression model.
explainability_logistic_regression = LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="saga",
    max_iter=5000,
    tol=1e-3,
    random_state=42,
    n_jobs=-1,
)


# Build and train the explainability pipeline.
explainability_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            explainability_preprocessor,
        ),
        (
            "classifier",
            explainability_logistic_regression,
        ),
    ]
)

explainability_pipeline.fit(
    X_explainability,
    y_explainability,
)


# Verify optimizer convergence.
classifier = explainability_pipeline.named_steps[
    "classifier"
]

iterations_used = int(classifier.n_iter_[0])

print(
    "Explainability Logistic Regression model "
    "trained successfully."
)
print(f"Training rows: {len(X_explainability):,}")
print(f"Raw predictors: {len(MODEL_INPUT_COLUMNS)}")
print(f"Iterations used: {iterations_used:,}")
print(
    "Converged:",
    iterations_used
    < explainability_logistic_regression.max_iter,
)

### 10.3 Generate SHAP Values

SHAP (SHapley Additive exPlanations) is applied to the explainability Logistic Regression model to quantify how each predictor contributes to the predicted probability of flight delay.

Both global and local explanations are generated.

Global explanations summarize feature importance across many flights, while local explanations explain why a particular flight received its predicted delay probability.

These explanations describe the behavior of the predictive model and should not be interpreted as evidence of causal relationships.

In [0]:
# Obtain encoded feature names after preprocessing.

encoded_feature_names = (
    explainability_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

clean_feature_names = [
    feature
    .replace("categorical__", "")
    .replace("numerical__", "")
    .replace("_", " ")
    for feature in encoded_feature_names
]

print(f"Encoded explainability features: {len(encoded_feature_names):,}")

In [0]:
clean_feature_names = [
    feature
    .replace("categorical__", "")
    .replace("numerical__", "")
    .replace("_", " ")
    for feature in encoded_feature_names
]

DISPLAY_FEATURE_NAMES = {
    "DISTANCE": "Flight Distance",
    "CRS ELAPSED TIME": "Scheduled Elapsed Time",
    "DEP HOUR": "Departure Hour",
    "CRS DEP TIME": "Scheduled Departure Time",
    "CRS ARR TIME": "Scheduled Arrival Time",
    "MONTH": "Month",
    "DAY OF WEEK": "Day of Week",
    "IS WEEKEND": "Weekend Flight",
    "ROUTE HIST DELAY RATE": "Historical Route Delay Rate",
    "AIRLINE HIST DELAY RATE": "Historical Airline Delay Rate",
    "ORIGIN HIST DELAY RATE": "Historical Origin Delay Rate",
    "DEST HIST DELAY RATE": "Historical Destination Delay Rate",
}

clean_feature_names = [
    DISPLAY_FEATURE_NAMES.get(feature, feature)
    for feature in clean_feature_names
]

print("Readable feature names prepared successfully.")

### 10.4 Compute SHAP Values

The trained explainability Logistic Regression model is interpreted using SHAP.

The generated SHAP values quantify the contribution of each encoded predictor to the predicted probability of flight delay.

These values are subsequently used to produce:

- Global feature importance
- SHAP summary plots
- Dependence plots
- Local explanations for individual flights displayed in the dashboard.

In [0]:
import shap
import pandas as pd

# ---------------------------------------------------------
# Transform the readable features using the preprocessing
# pipeline.
# ---------------------------------------------------------

X_transformed = (
    explainability_pipeline
    .named_steps["preprocessor"]
    .transform(X_explainability)
)

print("Transformed feature matrix:")
print(X_transformed.shape)

#### Initialize the SHAP Explainer

A SHAP `LinearExplainer` is initialized using the trained explainability Logistic Regression model and the transformed explainability feature matrix.

The explainer estimates the contribution of every encoded predictor to the predicted probability of flight delay. For computational efficiency, SHAP automatically samples a subset of background observations while preserving the overall feature distribution.

This initialization prepares the explainability model for both global and local interpretation.

In [0]:
classifier = (
    explainability_pipeline
    .named_steps["classifier"]
)

explainer = shap.LinearExplainer(
    classifier,
    X_transformed,
)

print("SHAP explainer created successfully.")

#### Compute SHAP Values

SHAP values are computed for every observation in the explainability dataset.

Each SHAP value represents the contribution of an individual encoded predictor toward increasing or decreasing the predicted probability of flight delay for a specific observation.

The resulting explanation matrix is subsequently used to generate global feature importance rankings, SHAP summary plots, dependence plots, and local explanations for individual flights displayed in the dashboard.

In [0]:
shap_values = explainer(X_transformed)

print("SHAP values computed successfully.")
print(shap_values.values.shape)

### 10.5 Global Feature Importance

Global SHAP analysis identifies the predictors that contribute most strongly to flight delay predictions across the entire explainability dataset.

The importance ranking is calculated using the mean absolute SHAP value of every encoded feature. Higher values indicate that a predictor has greater influence on the model's predictions.

In [0]:
import numpy as np
import pandas as pd

feature_importance = pd.DataFrame(
    {
        "Feature": clean_feature_names,
        "MeanAbsSHAP": np.abs(
            shap_values.values
        ).mean(axis=0),
    }
)

feature_importance = (
    feature_importance
    .sort_values(
        "MeanAbsSHAP",
        ascending=False,
    )
)

display(
    feature_importance.head(20)
)

### 10.6 SHAP Summary Plot

The SHAP summary plot visualizes both the magnitude and direction of feature effects across the explainability dataset.

Each point represents one flight. The horizontal position indicates the SHAP value, while the color represents the corresponding feature value.

Positive SHAP values increase the predicted probability of flight delay, whereas negative SHAP values reduce the predicted probability.

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,8))

shap.summary_plot(
    shap_values,
    X_transformed,
    feature_names=clean_feature_names,
    max_display=20,
    show=False,
)

plt.tight_layout()
plt.show()

### 10.7 Direction of Feature Effects

The direction of each feature's influence is evaluated using the average signed SHAP value.

A positive mean SHAP value indicates that the feature tends to increase predicted delay risk across the explainability sample. A negative mean SHAP value indicates that the feature tends to reduce predicted delay risk.

Because categorical predictors are one-hot encoded, the direction applies to the specific category shown in the feature name. These results describe model behavior and should not be interpreted as causal relationships.

In [0]:
direction_of_effects = pd.DataFrame(
    {
        "Feature": clean_feature_names,
        "Mean_SHAP": shap_values.values.mean(axis=0),
        "Mean_Absolute_SHAP": np.abs(
            shap_values.values
        ).mean(axis=0),
    }
)

direction_of_effects["Effect_Direction"] = np.where(
    direction_of_effects["Mean_SHAP"] > 0,
    "Increases predicted delay risk",
    np.where(
        direction_of_effects["Mean_SHAP"] < 0,
        "Decreases predicted delay risk",
        "Neutral average effect",
    ),
)

direction_of_effects = (
    direction_of_effects
    .sort_values(
        "Mean_Absolute_SHAP",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    direction_of_effects.head(20)
)

### 10.8 SHAP Dependence Plots

SHAP dependence plots illustrate how changes in individual predictor values influence the model's predicted probability of flight delay.

Unlike the global summary plot, each dependence plot focuses on a single predictor and shows both the magnitude and direction of its SHAP contribution across the explainability dataset.

The following plots are generated for the most influential operational predictors identified during global SHAP analysis:

- Flight Distance
- Historical Route Delay Rate
- Departure Hour

These plots support deeper interpretation of the model's behavior and help identify how important operational factors influence predicted delay risk.

In [0]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import sparse


DEPENDENCE_FEATURES = [
    "Flight Distance",
    "Historical Route Delay Rate",
    "Departure Hour",
]


for feature_name in DEPENDENCE_FEATURES:

    if feature_name not in clean_feature_names:
        print(f"Skipping {feature_name}: feature not found.")
        continue

    feature_index = clean_feature_names.index(feature_name)

    # Extract only the selected feature column.
    feature_column = X_transformed[:, feature_index]

    if sparse.issparse(feature_column):
        feature_values = feature_column.toarray().ravel()
    else:
        feature_values = np.asarray(feature_column).ravel()

    feature_shap_values = shap_values.values[:, feature_index]

    # Remove any non-finite observations before plotting.
    valid_rows = (
        np.isfinite(feature_values)
        & np.isfinite(feature_shap_values)
    )

    plt.figure(figsize=(9, 6))

    plt.scatter(
        feature_values[valid_rows],
        feature_shap_values[valid_rows],
        alpha=0.35,
        s=18,
    )

    plt.axhline(
        y=0,
        linewidth=1,
        linestyle="--",
    )

    plt.title(
        f"SHAP Dependence Plot: {feature_name}"
    )
    plt.xlabel(f"{feature_name} (standardized value)")
    plt.ylabel("SHAP value (impact on model output)")

    plt.tight_layout()
    plt.show()

### 10.9 Local SHAP Explanations

Local SHAP explanations describe how individual predictor variables contributed to the predicted delay probability for a single flight.

Unlike the global explanations, which summarize model behavior across many observations, local explanations identify the specific factors that increased or decreased the predicted delay risk for an individual prediction.

These explanations support transparent decision-making by showing why a particular flight received its predicted delay probability.

In [0]:
import numpy as np
import shap


LOCAL_FLIGHT_INDEX = 0

# Obtain the surrogate model's predicted delay probability.
local_predicted_probability = (
    explainability_pipeline
    .predict_proba(
        X_explainability.iloc[
            [LOCAL_FLIGHT_INDEX]
        ]
    )[0, 1]
)

local_predicted_class = int(
    local_predicted_probability >= 0.50
)

# Build a readable SHAP Explanation object.
local_explanation = shap.Explanation(
    values=shap_values.values[
        LOCAL_FLIGHT_INDEX
    ],
    base_values=shap_values.base_values[
        LOCAL_FLIGHT_INDEX
    ],
    data=(
        X_transformed[
            LOCAL_FLIGHT_INDEX
        ]
        .toarray()
        .ravel()
    ),
    feature_names=clean_feature_names,
)

print(f"Selected flight index: {LOCAL_FLIGHT_INDEX}")
print(
    "Predicted delay probability: "
    f"{local_predicted_probability:.2%}"
)
print(
    "Predicted class: "
    f"{'Delayed' if local_predicted_class == 1 else 'On-time'}"
)

shap.plots.waterfall(
    local_explanation,
    max_display=15,
)

### Local Explanation Interpretation

The waterfall plot explains the surrogate model's prediction for one selected flight.

Red contributions increased the predicted delay risk, while blue contributions reduced it. The strongest risk-increasing factors for this flight included flight distance, the selected destination, and the historical route delay rate. Scheduled elapsed time, season, weekend status, and other operational characteristics reduced the model's predicted risk.

The SHAP values describe how the model formed this prediction. They do not demonstrate that the listed variables directly caused the flight outcome.

For the selected flight, Flight Distance provided the largest positive contribution toward the predicted delay probability. The selected destination (LGA), destination city (New York, NY), and Historical Route Delay Rate also increased the predicted delay risk.

Conversely, Scheduled Elapsed Time, the Winter season, Weekend Flight status, and the Origin State contributed toward reducing the predicted delay probability.

These SHAP values describe how the model reached its prediction for this individual flight. They should not be interpreted as evidence that the listed variables directly caused the flight outcome.